In [1]:
%pip install torch_geometric
%pip install pandas
%pip install torch
%pip install numpy
%pip install scikit-learn

Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: http://mirrors.aliyun.com/pypi/simple
Note: you may need to restart the kernel to use updated packages.


# German

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import time
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch import nn

from torch_geometric.utils import to_undirected, remove_self_loops, coalesce

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

# =========================================================
# 0) Reproducibility
# =========================================================
def set_all_seeds(seed: int = 42, deterministic: bool = True):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


# =========================================================
# Global config
# =========================================================
CSV_PATH = "/root/autodl-tmp/german/german.csv"
EDGE_PATH = "/root/autodl-tmp/german/german_edges.txt"
SAVE_DIR = "/root/autodl-tmp/german/best_parameter_run_fixed_split"

FIXED_SPLIT_SEED = 42
TRAIN_SEEDS = [43, 54, 59, 62, 67, 71, 83, 84, 96, 97]

# 我替你构思的一组 German 固定超参数
FIXED_PARAMS = {
    "hidden": 128,
    "num_layers": 6,
    "epochs": 300,
    "lr": 0.005,
    "weight_decay": 0.0005,
    "dropout": 0.2,
    "residual_alpha_deep": 0.1,
    "lambda_weak": 0.01,
    "lambda_mid": 0.05,
    "lambda_strong": 0.10,
    "fairness_warmup": 100,
    "use_rbf_mmd": False,
    "mmd_sigma": 1.0,
    "pos_weight_multiplier": 1.0,
}


# =========================================================
# 1) Small utilities
# =========================================================
def print_header(title):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def undirected_unique_edge_count(edge_index, num_nodes):
    u = torch.minimum(edge_index[0], edge_index[1])
    v = torch.maximum(edge_index[0], edge_index[1])
    uv = torch.stack([u, v], dim=0)
    uv = coalesce(uv, None, num_nodes, num_nodes)[0]
    return uv.size(1)


def safe_mean(arr):
    arr = np.array(arr, dtype=float)
    return float(np.nanmean(arr))


def safe_std(arr):
    arr = np.array(arr, dtype=float)
    return float(np.nanstd(arr, ddof=1)) if len(arr) > 1 else 0.0


# =========================================================
# 2) Loader
# =========================================================
def read_and_validate_edges(edge_file, n, make_undirected=True, drop_self_loops=True):
    edges = np.genfromtxt(edge_file).astype(int)
    if edges.ndim == 1 and edges.size == 2:
        edges = edges.reshape(1, 2)

    mask = (
        (edges[:, 0] >= 0) & (edges[:, 0] < n) &
        (edges[:, 1] >= 0) & (edges[:, 1] < n)
    )
    if (~mask).any():
        print(f"Edge bounds violations removed: {np.sum(~mask)}")
    edges = edges[mask]

    edge_index = torch.tensor(edges.T, dtype=torch.long)
    if drop_self_loops:
        edge_index, _ = remove_self_loops(edge_index)
    if make_undirected:
        edge_index = to_undirected(edge_index, num_nodes=n)

    edge_index = coalesce(edge_index, None, n, n)[0]
    return edge_index


def stratified_masks(y, train_ratio=0.5, val_ratio=0.25, seed=42):
    y_np = y.cpu().numpy()
    N = len(y_np)
    tv_ratio = train_ratio + val_ratio

    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=1 - tv_ratio, random_state=seed)
    tv_idx, te_idx = next(sss1.split(np.zeros(N), y_np))

    y_tv = y_np[tv_idx]
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=val_ratio / tv_ratio, random_state=seed)
    tr_rel, va_rel = next(sss2.split(np.zeros(len(tv_idx)), y_tv))

    tr_idx = tv_idx[tr_rel]
    va_idx = tv_idx[va_rel]

    train_mask = torch.zeros(N, dtype=torch.bool)
    val_mask = torch.zeros(N, dtype=torch.bool)
    test_mask = torch.zeros(N, dtype=torch.bool)

    train_mask[tr_idx] = True
    val_mask[va_idx] = True
    test_mask[te_idx] = True
    return {"train": train_mask, "val": val_mask, "test": test_mask}


def load_german_dataset(
    csv_path=CSV_PATH,
    edge_path=EDGE_PATH,
    train_ratio=0.5,
    val_ratio=0.25,
    split_seed=42,
):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Missing file: {csv_path}")
    if not os.path.exists(edge_path):
        raise FileNotFoundError(f"Missing file: {edge_path}")

    print_header(f"Loading German dataset | fixed split seed={split_seed}")
    t0 = time.time()
    df = pd.read_csv(csv_path)

    predict_attr = "GoodCustomer"
    sens_attr = "Gender"

    unique_lbls = pd.unique(df[predict_attr])
    print("Unique labels in CSV:", unique_lbls)

    labels = df[predict_attr].to_numpy(copy=True)
    labels = np.where(labels == -1, 0, labels).astype(int)
    if not set(np.unique(labels)).issubset({0, 1}):
        raise ValueError("Labels must map to {0,1} after conversion.")

    mapping = {"Female": 1, "Male": 0, "female": 1, "male": 0, "F": 1, "M": 0}
    sens_series = df[sens_attr].astype(str).map(mapping)
    sens_series = pd.to_numeric(sens_series, errors="coerce").fillna(0).astype(int)
    sens = sens_series.values.astype(int)

    labels_t = torch.tensor(labels, dtype=torch.long)
    sens_t = torch.tensor(sens, dtype=torch.long)
    masks = stratified_masks(labels_t, train_ratio=train_ratio, val_ratio=val_ratio, seed=split_seed)

    drop_cols = [predict_attr, sens_attr, "OtherLoansAtStore", "PurposeOfLoan"]
    keep_cols = [c for c in df.columns if c not in drop_cols]
    X_df = df[keep_cols].copy()

    cat_cols = [c for c in X_df.columns if X_df[c].dtype == "object"]
    if cat_cols:
        X_df = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)

    for c in X_df.columns:
        X_df[c] = pd.to_numeric(X_df[c], errors="coerce")
    X_df = X_df.fillna(0)

    X_np = X_df.to_numpy(dtype=np.float32, copy=True)

    # train-only standardization
    train_idx = masks["train"].cpu().numpy()
    scaler = StandardScaler()
    scaler.fit(X_np[train_idx])
    X_np = scaler.transform(X_np).astype(np.float32)

    features = torch.tensor(X_np, dtype=torch.float32)

    n = features.size(0)
    edge_index = read_and_validate_edges(edge_path, n, make_undirected=True, drop_self_loops=True)

    print("Num nodes:", n, " Num features:", features.size(1), " Num edges:", edge_index.size(1))
    deg = torch.bincount(edge_index[0], minlength=n).cpu().numpy()
    print("Degree: min/med/max:", int(deg.min()), float(np.median(deg)), int(deg.max()))
    print("Isolated nodes:", int((deg == 0).sum()))

    u_lbl, c_lbl = np.unique(labels, return_counts=True)
    print("Label distribution:", dict(zip(u_lbl.tolist(), c_lbl.tolist())))

    u_s, c_s = np.unique(sens, return_counts=True)
    print("Sensitive distribution (0=Male,1=Female):", dict(zip(u_s.tolist(), c_s.tolist())))

    overlap = {
        "train∩val": int((masks["train"] & masks["val"]).sum()),
        "train∩test": int((masks["train"] & masks["test"]).sum()),
        "val∩test": int((masks["val"] & masks["test"]).sum()),
    }
    print("Mask overlaps (should be 0s):", overlap)
    print("Train count:", int(masks["train"].sum()))
    print("Val count:", int(masks["val"].sum()))
    print("Test count:", int(masks["test"].sum()))

    print(f"Done in {time.time()-t0:.3f}s")

    num_nodes = features.size(0)
    m_dir = edge_index.size(1)
    m_undir = undirected_unique_edge_count(edge_index, num_nodes)
    print(f"Edges (directed columns): {m_dir}")
    print(f"Edges (undirected unique pairs): {m_undir}")

    return edge_index, features, labels_t, sens_t, masks


# =========================================================
# 3) Graph normalization
# =========================================================
def build_gcn_norm(edge_index, num_nodes, add_self_loops=True, device=None):
    if device is None:
        device = edge_index.device

    row, col = edge_index
    if add_self_loops:
        self_loops = torch.arange(num_nodes, device=device)
        self_loops = torch.stack([self_loops, self_loops], dim=0)
        edge_index = torch.cat([edge_index, self_loops], dim=1)
        edge_index = coalesce(edge_index, None, num_nodes, num_nodes)[0]
        row, col = edge_index

    deg = torch.bincount(row, minlength=num_nodes).float()
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0

    norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
    return edge_index, norm


def spmm_mean(edge_index, edge_weight, x, num_nodes):
    row, col = edge_index
    out = torch.zeros_like(x)
    msg = x[col] * edge_weight.unsqueeze(-1)
    out.index_add_(0, row, msg)
    return out


# =========================================================
# 4) Model
# =========================================================
class DepthFairLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.2, use_bn=True):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=False)
        self.bn = nn.BatchNorm1d(out_dim) if use_bn else nn.Identity()
        self.do = nn.Dropout(dropout)
        self.res_proj = nn.Linear(in_dim, out_dim, bias=False) if in_dim != out_dim else None

    def forward(self, x, edge_index, edge_weight, identity=None, residual_alpha=0.0):
        h_in = x
        x = self.lin(x)
        h = spmm_mean(edge_index, edge_weight, x, num_nodes=x.size(0))

        h = h + (self.res_proj(h_in) if self.res_proj is not None else h_in)

        if identity is not None and residual_alpha > 0:
            h = h + residual_alpha * identity

        h = self.bn(h)
        h = F.relu(h)
        h = self.do(h)
        return h


class DepthFairGNN(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden=128,
        out_dim=2,
        num_layers=6,
        dropout=0.2,
        residual_alpha_deep=0.2,
    ):
        super().__init__()
        assert num_layers >= 2

        self.residual_alpha_deep = residual_alpha_deep
        self.input_proj = nn.Linear(in_dim, hidden, bias=False)

        self.layers = nn.ModuleList()
        self.layers.append(DepthFairLayer(in_dim, hidden, dropout=dropout))
        for _ in range(num_layers - 2):
            self.layers.append(DepthFairLayer(hidden, hidden, dropout=dropout))

        self.cls = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x, edge_index, edge_weight):
        hidden_states = []
        identity = self.input_proj(x)

        h = x
        for i, layer in enumerate(self.layers):
            layer_num = i + 1
            alpha = self.residual_alpha_deep if layer_num >= 5 else 0.0
            h = layer(h, edge_index, edge_weight, identity=identity, residual_alpha=alpha)
            hidden_states.append(h)

        logits = self.cls(h)
        return logits, hidden_states


# =========================================================
# 5) Fairness loss + metrics
# =========================================================
def linear_mmd(x0, x1):
    if x0.size(0) == 0 or x1.size(0) == 0:
        device = x0.device if x0.numel() > 0 else x1.device
        return torch.tensor(0.0, device=device)
    return ((x0.mean(dim=0) - x1.mean(dim=0)) ** 2).mean()


def rbf_mmd(x0, x1, sigma=1.0):
    if x0.size(0) < 2 or x1.size(0) < 2:
        device = x0.device if x0.numel() > 0 else x1.device
        return torch.tensor(0.0, device=device)

    xx = torch.cdist(x0, x0, p=2).pow(2)
    yy = torch.cdist(x1, x1, p=2).pow(2)
    xy = torch.cdist(x0, x1, p=2).pow(2)

    k_xx = torch.exp(-xx / (2 * sigma * sigma))
    k_yy = torch.exp(-yy / (2 * sigma * sigma))
    k_xy = torch.exp(-xy / (2 * sigma * sigma))
    return k_xx.mean() + k_yy.mean() - 2 * k_xy.mean()


def depth_schedule(num_layers, lambda_weak=0.01, lambda_mid=0.05, lambda_strong=0.10):
    weights = []
    for l in range(1, num_layers + 1):
        if l <= 2:
            weights.append(lambda_weak)
        elif l <= 4:
            weights.append(lambda_mid)
        else:
            weights.append(lambda_strong)
    return weights


def depth_fairness_loss(hidden_states, sensitive, train_mask, layer_weights, use_rbf=False, sigma=1.0):
    loss = torch.tensor(0.0, device=hidden_states[0].device)
    for l, h in enumerate(hidden_states, start=1):
        h_train = h[train_mask]
        s_train = sensitive[train_mask]
        h0 = h_train[s_train == 0]
        h1 = h_train[s_train == 1]

        fair_l = rbf_mmd(h0, h1, sigma=sigma) if use_rbf else linear_mmd(h0, h1)
        loss = loss + layer_weights[l - 1] * fair_l
    return loss


@torch.no_grad()
def metrics(logits, y, sensitive):
    pred = logits.argmax(dim=-1)
    acc = (pred == y).float().mean().item()

    p1 = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
    y_np = y.cpu().numpy()
    s_np = sensitive.cpu().numpy()
    pred_np = pred.cpu().numpy()

    try:
        auc = roc_auc_score(y_np, p1)
    except ValueError:
        auc = float("nan")

    f1 = f1_score(y_np, pred_np, zero_division=0)
    precision = precision_score(y_np, pred_np, zero_division=0)
    recall = recall_score(y_np, pred_np, zero_division=0)
    pred_pos_rate = float((pred_np == 1).mean())

    s0 = s_np == 0
    s1 = s_np == 1
    p1_mean_s0 = p1[s0].mean() if s0.any() else np.nan
    p1_mean_s1 = p1[s1].mean() if s1.any() else np.nan
    sp = abs(p1_mean_s0 - p1_mean_s1)

    pred_pos = (pred_np == 1)
    y_pos = (y_np == 1)

    def tpr(mask):
        denom = (y_pos & mask).sum()
        return (pred_pos & y_pos & mask).sum() / denom if denom > 0 else np.nan

    tpr0 = tpr(s0)
    tpr1 = tpr(s1)
    eo = abs(tpr0 - tpr1) if np.isfinite(tpr0) and np.isfinite(tpr1) else np.nan

    return {
        "acc": float(acc),
        "auc": float(auc) if not np.isnan(auc) else np.nan,
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "pred_pos_rate": float(pred_pos_rate),
        "sp": float(sp) if not np.isnan(sp) else np.nan,
        "eo": float(eo) if not np.isnan(eo) else np.nan,
    }


# =========================================================
# 6) Baseline
# =========================================================
def baseline_logreg(features, labels, masks, sensitive):
    print_header("Baseline: Logistic Regression (features-only)")
    X = features.cpu().numpy()
    y = labels.cpu().numpy()
    s_np = sensitive.cpu().numpy()

    tr = masks["train"].cpu().numpy()
    va = masks["val"].cpu().numpy()
    te = masks["test"].cpu().numpy()

    def fairness_metrics(prob, pred, y_true, sens):
        s0, s1 = sens == 0, sens == 1
        p1_s0 = prob[s0].mean() if s0.any() else np.nan
        p1_s1 = prob[s1].mean() if s1.any() else np.nan
        sp = abs(p1_s0 - p1_s1)

        y_pos = y_true == 1
        pred_pos = pred == 1

        def tpr(mask):
            denom = (y_pos & mask).sum()
            return (pred_pos & y_pos & mask).sum() / denom if denom > 0 else np.nan

        tpr0 = tpr(s0)
        tpr1 = tpr(s1)
        eo = abs(tpr0 - tpr1) if np.isfinite(tpr0) and np.isfinite(tpr1) else np.nan
        return sp, eo

    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(X[tr], y[tr])

    for name, msk in [("Train", tr), ("Val", va), ("Test", te)]:
        prob = clf.predict_proba(X[msk])[:, 1]
        pred = (prob >= 0.5).astype(int)
        acc = accuracy_score(y[msk], pred)
        try:
            auc = roc_auc_score(y[msk], prob)
        except ValueError:
            auc = float("nan")
        sp, eo = fairness_metrics(prob, pred, y[msk], s_np[msk])
        print(f"{name}: ACC={acc:.4f} AUC={auc:.4f} SP={sp:.4f} EO={eo:.4f}")

    return clf


# =========================================================
# 7) Train loop
# =========================================================
def train_one_seed(
    x, edge_index, y, sensitive, masks,
    hidden=128,
    num_layers=6,
    epochs=300,
    lr=0.005,
    weight_decay=5e-4,
    dropout=0.2,
    residual_alpha_deep=0.1,
    lambda_weak=0.01,
    lambda_mid=0.05,
    lambda_strong=0.10,
    fairness_warmup=100,
    use_rbf_mmd=False,
    mmd_sigma=1.0,
    pos_weight_multiplier=1.0,
    seed=42,
):
    set_all_seeds(seed, deterministic=True)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    x = x.to(device)
    edge_index = edge_index.to(device)
    y = y.to(device)
    sensitive = sensitive.to(device)

    train_mask = masks["train"].to(device)
    val_mask = masks["val"].to(device)
    test_mask = masks["test"].to(device)

    edge_index_norm, edge_weight = build_gcn_norm(
        edge_index, num_nodes=x.size(0), add_self_loops=True, device=device
    )

    # train-only class weight
    y_train = y[train_mask]
    num_pos = int((y_train == 1).sum().item())
    num_neg = int((y_train == 0).sum().item())
    base_pos_weight = num_neg / max(num_pos, 1)
    final_pos_weight = base_pos_weight * pos_weight_multiplier
    cw = torch.tensor([1.0, final_pos_weight], device=device, dtype=torch.float32)

    print_header("CLASS WEIGHT INFO")
    print(f"train neg = {num_neg}")
    print(f"train pos = {num_pos}")
    print(f"base_pos_weight = {base_pos_weight:.4f}")
    print(f"pos_weight_multiplier = {pos_weight_multiplier:.4f}")
    print(f"final positive class weight = {final_pos_weight:.4f}")

    model = DepthFairGNN(
        in_dim=x.size(1),
        hidden=hidden,
        out_dim=2,
        num_layers=num_layers,
        dropout=dropout,
        residual_alpha_deep=residual_alpha_deep
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    base_layer_weights = depth_schedule(
        num_layers=num_layers,
        lambda_weak=lambda_weak,
        lambda_mid=lambda_mid,
        lambda_strong=lambda_strong
    )

    best_val_score = -1e18
    best_state = None
    best_epoch = 0
    best_val_metrics = None
    best_test_metrics = None

    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()

        logits, hidden_states = model(x, edge_index_norm, edge_weight)

        alpha = min(1.0, ep / float(fairness_warmup)) if fairness_warmup > 0 else 1.0
        layer_weights = [alpha * w for w in base_layer_weights]

        ce = F.cross_entropy(logits[train_mask], y[train_mask], weight=cw)
        fair = depth_fairness_loss(
            hidden_states=hidden_states,
            sensitive=sensitive,
            train_mask=train_mask,
            layer_weights=layer_weights,
            use_rbf=use_rbf_mmd,
            sigma=mmd_sigma
        )

        loss = ce + fair
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            logits_eval, _ = model(x, edge_index_norm, edge_weight)
            trm = metrics(logits_eval[train_mask], y[train_mask], sensitive[train_mask])
            valm = metrics(logits_eval[val_mask], y[val_mask], sensitive[val_mask])
            tem = metrics(logits_eval[test_mask], y[test_mask], sensitive[test_mask])

            val_score = valm["auc"] + 0.2 * valm["acc"] - 0.5 * valm["sp"]

            if val_score > best_val_score:
                best_val_score = val_score
                best_epoch = ep
                best_val_metrics = valm.copy()
                best_test_metrics = tem.copy()
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if ep % 10 == 0 or ep == 1:
            print(
                f"[{ep:03d}] loss={loss.item():.4f} ce={ce.item():.4f} fair={fair.item():.4f} "
                f"train(acc={trm['acc']:.4f}, auc={trm['auc']:.4f}, sp={trm['sp']:.4f}, eo={trm['eo']:.4f}) "
                f"val(acc={valm['acc']:.4f}, auc={valm['auc']:.4f}, sp={valm['sp']:.4f}, eo={valm['eo']:.4f}) "
                f"test(acc={tem['acc']:.4f}, auc={tem['auc']:.4f}, sp={tem['sp']:.4f}, eo={tem['eo']:.4f})"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        logits, _ = model(x, edge_index_norm, edge_weight)
        train_metrics = metrics(logits[train_mask], y[train_mask], sensitive[train_mask])
        val_metrics = metrics(logits[val_mask], y[val_mask], sensitive[val_mask])
        test_metrics = metrics(logits[test_mask], y[test_mask], sensitive[test_mask])

    print_header("BEST MODEL SUMMARY")
    print(f"Best epoch: {best_epoch}")
    print(f"Best val score: {best_val_score:.6f}")

    print_header("FINAL TRAIN METRICS")
    print(train_metrics)

    print_header("FINAL VAL METRICS")
    print(val_metrics)

    print_header("FINAL TEST METRICS")
    print(test_metrics)

    return {
        "best_epoch": best_epoch,
        "best_val_score": best_val_score,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }


# =========================================================
# 8) Multi-seed evaluation
# =========================================================
def run_fixed_best_10seeds_fixed_split_german():
    os.makedirs(SAVE_DIR, exist_ok=True)

    print_header("Fixed hyperparameters")
    print(FIXED_PARAMS)

    print_header("Evaluation setting")
    print(f"Fixed split seed: {FIXED_SPLIT_SEED}")
    print(f"Training seeds: {TRAIN_SEEDS}")

    # fixed split: load once
    set_all_seeds(FIXED_SPLIT_SEED, deterministic=True)
    edge_index, X, y, s, masks = load_german_dataset(
        csv_path=CSV_PATH,
        edge_path=EDGE_PATH,
        split_seed=FIXED_SPLIT_SEED,
    )

    _ = baseline_logreg(X, y, masks, s)

    seed_rows = []

    for train_seed in TRAIN_SEEDS:
        print_header(f"Training with train_seed = {train_seed}")

        result = train_one_seed(
            X, edge_index, y, s, masks,
            hidden=FIXED_PARAMS["hidden"],
            num_layers=FIXED_PARAMS["num_layers"],
            epochs=FIXED_PARAMS["epochs"],
            lr=FIXED_PARAMS["lr"],
            weight_decay=FIXED_PARAMS["weight_decay"],
            dropout=FIXED_PARAMS["dropout"],
            residual_alpha_deep=FIXED_PARAMS["residual_alpha_deep"],
            lambda_weak=FIXED_PARAMS["lambda_weak"],
            lambda_mid=FIXED_PARAMS["lambda_mid"],
            lambda_strong=FIXED_PARAMS["lambda_strong"],
            fairness_warmup=FIXED_PARAMS["fairness_warmup"],
            use_rbf_mmd=FIXED_PARAMS["use_rbf_mmd"],
            mmd_sigma=FIXED_PARAMS["mmd_sigma"],
            pos_weight_multiplier=FIXED_PARAMS["pos_weight_multiplier"],
            seed=train_seed,
        )

        row = {
            "train_seed": train_seed,
            "best_epoch": result["best_epoch"],
            "best_val_score": result["best_val_score"],

            "ACC": result["test_metrics"]["acc"],
            "SP": result["test_metrics"]["sp"],
            "AUC": result["test_metrics"]["auc"],
            "EO": result["test_metrics"]["eo"],

            "F1": result["test_metrics"]["f1"],
            "Precision": result["test_metrics"]["precision"],
            "Recall": result["test_metrics"]["recall"],
            "PredPosRate": result["test_metrics"]["pred_pos_rate"],
        }
        seed_rows.append(row)
        print("Seed result:", row)

    seed_df = pd.DataFrame(seed_rows)

    summary_df = pd.DataFrame([{
        "ACC_mean": safe_mean(seed_df["ACC"]),
        "ACC_std": safe_std(seed_df["ACC"]),
        "SP_mean": safe_mean(seed_df["SP"]),
        "SP_std": safe_std(seed_df["SP"]),
        "AUC_mean": safe_mean(seed_df["AUC"]),
        "AUC_std": safe_std(seed_df["AUC"]),
        "EO_mean": safe_mean(seed_df["EO"]),
        "EO_std": safe_std(seed_df["EO"]),
        "F1_mean": safe_mean(seed_df["F1"]),
        "F1_std": safe_std(seed_df["F1"]),
        "Precision_mean": safe_mean(seed_df["Precision"]),
        "Precision_std": safe_std(seed_df["Precision"]),
        "Recall_mean": safe_mean(seed_df["Recall"]),
        "Recall_std": safe_std(seed_df["Recall"]),
        "PredPosRate_mean": safe_mean(seed_df["PredPosRate"]),
        "PredPosRate_std": safe_std(seed_df["PredPosRate"]),
    }])

    print_header("Per-seed results")
    print(seed_df[["train_seed", "ACC", "SP", "AUC", "EO", "F1", "Precision", "Recall", "PredPosRate"]])

    print_header("Final mean/std")
    print(summary_df)

    results_csv = os.path.join(SAVE_DIR, "seed_results.csv")
    results_json = os.path.join(SAVE_DIR, "seed_results.json")
    summary_csv = os.path.join(SAVE_DIR, "summary_mean_std.csv")
    summary_json = os.path.join(SAVE_DIR, "summary_mean_std.json")

    seed_df.to_csv(results_csv, index=False)
    seed_df.to_json(results_json, orient="records", indent=2)
    summary_df.to_csv(summary_csv, index=False)
    summary_df.to_json(summary_json, orient="records", indent=2)

    print(f"\nSaved per-seed results to: {results_csv}")
    print(f"Saved per-seed results json to: {results_json}")
    print(f"Saved summary csv to: {summary_csv}")
    print(f"Saved summary json to: {summary_json}")

    return seed_df, summary_df


# =========================================================
# 9) Main
# =========================================================
if __name__ == "__main__":
    run_fixed_best_10seeds_fixed_split_german()


Fixed hyperparameters
{'hidden': 128, 'num_layers': 6, 'epochs': 300, 'lr': 0.005, 'weight_decay': 0.0005, 'dropout': 0.2, 'residual_alpha_deep': 0.1, 'lambda_weak': 0.01, 'lambda_mid': 0.05, 'lambda_strong': 0.1, 'fairness_warmup': 100, 'use_rbf_mmd': False, 'mmd_sigma': 1.0, 'pos_weight_multiplier': 1.0}

Evaluation setting
Fixed split seed: 42
Training seeds: [43, 54, 59, 62, 67, 71, 83, 84, 96, 97]

Loading German dataset | fixed split seed=42
Unique labels in CSV: [ 1 -1]
Num nodes: 1000  Num features: 26  Num edges: 43484
Degree: min/med/max: 5 37.0 286
Isolated nodes: 0
Label distribution: {0: 300, 1: 700}
Sensitive distribution (0=Male,1=Female): {0: 690, 1: 310}
Mask overlaps (should be 0s): {'train∩val': 0, 'train∩test': 0, 'val∩test': 0}
Train count: 500
Val count: 250
Test count: 250
Done in 0.082s
Edges (directed columns): 43484
Edges (undirected unique pairs): 21742

Baseline: Logistic Regression (features-only)
Train: ACC=0.6840 AUC=0.7544 SP=0.0686 EO=0.0906
Val: ACC=0

# Bail

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import time
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch import nn

# PyG utilities used ONLY for graph helpers (no Data object)
from torch_geometric.utils import to_undirected, remove_self_loops, coalesce

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score


# =========================================================
# 0) Reproducibility
# =========================================================
def set_all_seeds(seed: int, deterministic: bool = True):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True)
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


# =========================================================
# 1) Small utilities
# =========================================================
def print_header(title):
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)

def undirected_unique_edge_count(edge_index, num_nodes):
    u = torch.minimum(edge_index[0], edge_index[1])
    v = torch.maximum(edge_index[0], edge_index[1])
    uv = torch.stack([u, v], dim=0)
    uv = coalesce(uv, None, num_nodes, num_nodes)[0]
    return uv.size(1)


# =========================================================
# 2) Loader  ->  edge_index, features, labels, sens, masks
# =========================================================
def read_and_validate_edges(edge_file, n, make_undirected=True, drop_self_loops=True):
    edges = np.genfromtxt(edge_file).astype(int)

    if edges.ndim == 1 and edges.size == 2:
        edges = edges.reshape(1, 2)

    if edges.ndim != 2 or edges.shape[1] != 2:
        raise ValueError(f"Edge file must contain two columns, got shape {edges.shape}")

    mask = (
        (edges[:, 0] >= 0) & (edges[:, 0] < n) &
        (edges[:, 1] >= 0) & (edges[:, 1] < n)
    )
    if (~mask).any():
        print(f"Edge bounds violations removed: {np.sum(~mask)}")
    edges = edges[mask]

    edge_index = torch.tensor(edges.T, dtype=torch.long)

    if drop_self_loops:
        edge_index, _ = remove_self_loops(edge_index)
    if make_undirected:
        edge_index = to_undirected(edge_index, num_nodes=n)

    edge_index = coalesce(edge_index, None, n, n)[0]
    return edge_index


def bail_fixed_train_masks(y, train_per_class=100, val_ratio=0.25, test_ratio=0.25, seed=42):
    """
    Bail split protocol:
    1) For training set, randomly sample `train_per_class` nodes from each class.
    2) From the remaining nodes, sample validation and test sets so that:
         val size  ~= val_ratio  * N
         test size ~= test_ratio * N
       with stratification on labels.
    """
    y_np = y.cpu().numpy()
    N = len(y_np)

    assert (val_ratio + test_ratio) < 1.0, "val_ratio + test_ratio must be < 1"
    assert set(np.unique(y_np)).issubset({0, 1}), "This function assumes binary labels {0,1}"

    rng = np.random.RandomState(seed)

    class0_idx = np.where(y_np == 0)[0]
    class1_idx = np.where(y_np == 1)[0]

    if len(class0_idx) < train_per_class or len(class1_idx) < train_per_class:
        raise ValueError(
            f"Not enough samples for fixed-per-class training split. "
            f"class0={len(class0_idx)}, class1={len(class1_idx)}, train_per_class={train_per_class}"
        )

    train0 = rng.choice(class0_idx, size=train_per_class, replace=False)
    train1 = rng.choice(class1_idx, size=train_per_class, replace=False)
    train_idx = np.concatenate([train0, train1])
    rng.shuffle(train_idx)

    train_set = set(train_idx.tolist())
    remain_idx = np.array([i for i in range(N) if i not in train_set], dtype=int)
    y_remain = y_np[remain_idx]

    val_target = int(round(val_ratio * N))
    test_target = int(round(test_ratio * N))

    if val_target + test_target > len(remain_idx):
        raise ValueError(
            f"Not enough remaining nodes for val/test split. "
            f"remaining={len(remain_idx)}, val_target={val_target}, test_target={test_target}"
        )

    sss_test = StratifiedShuffleSplit(
        n_splits=1,
        test_size=test_target,
        random_state=seed
    )
    remain_for_val_idx_rel, test_idx_rel = next(sss_test.split(np.zeros(len(remain_idx)), y_remain))

    pool_for_val = remain_idx[remain_for_val_idx_rel]
    test_idx = remain_idx[test_idx_rel]

    y_pool_for_val = y_np[pool_for_val]

    sss_val = StratifiedShuffleSplit(
        n_splits=1,
        test_size=val_target,
        random_state=seed
    )
    _, val_idx_rel = next(sss_val.split(np.zeros(len(pool_for_val)), y_pool_for_val))
    val_idx = pool_for_val[val_idx_rel]

    train_mask = torch.zeros(N, dtype=torch.bool)
    val_mask   = torch.zeros(N, dtype=torch.bool)
    test_mask  = torch.zeros(N, dtype=torch.bool)

    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True

    return {"train": train_mask, "val": val_mask, "test": test_mask}


def load_bail_dataset(
    csv_path="/root/autodl-tmp/bail/bail.csv",
    edge_path="/root/autodl-tmp/bail/bail_edges.txt",
    standardize=True,
    drop_cols=("RECID", "WHITE", "FILE", "TIME"),
    train_per_class=100,
    split_seed=42,
):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Missing file: {csv_path}")
    if not os.path.exists(edge_path):
        raise FileNotFoundError(f"Missing file: {edge_path}")

    print_header(f"Loading bail dataset | fixed split seed={split_seed}")
    t0 = time.time()
    df = pd.read_csv(csv_path)

    predict_attr = "RECID"
    sens_attr = "WHITE"

    unique_lbls = pd.unique(df[predict_attr])
    print("Unique labels in CSV:", unique_lbls)

    labels = df[predict_attr].to_numpy(copy=True)
    labels = np.where(labels > 1, 1, labels).astype(int)

    if not set(np.unique(labels)).issubset({0, 1}):
        raise ValueError("Labels must map to {0,1} after conversion.")

    sens = df[sens_attr].to_numpy(copy=True).astype(int)
    if not set(np.unique(sens)).issubset({0, 1}):
        raise ValueError("Sensitive attribute must be binary {0,1} for this setup.")

    keep_cols = [c for c in df.columns if c not in drop_cols]
    X_df = df[keep_cols].copy()

    cat_cols = [c for c in X_df.columns if X_df[c].dtype == "object"]
    if cat_cols:
        X_df = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)

    X_np = X_df.to_numpy(dtype=np.float32)
    if standardize:
        scaler = StandardScaler()
        X_np = scaler.fit_transform(X_np).astype(np.float32)

    features = torch.tensor(X_np, dtype=torch.float32)
    labels_t = torch.tensor(labels, dtype=torch.long)
    sens_t   = torch.tensor(sens, dtype=torch.long)

    n = features.size(0)
    edge_index = read_and_validate_edges(edge_path, n, make_undirected=True, drop_self_loops=True)

    print("Num nodes:", n, " Num features:", features.size(1), " Num edges:", edge_index.size(1))
    deg = torch.bincount(edge_index[0], minlength=n).cpu().numpy()
    print("Degree: min/med/max:", int(deg.min()), float(np.median(deg)), int(deg.max()))
    print("Isolated nodes:", int((deg == 0).sum()))

    u_lbl, c_lbl = np.unique(labels, return_counts=True)
    print("Label distribution:", dict(zip(u_lbl.tolist(), c_lbl.tolist())))

    u_s, c_s = np.unique(sens, return_counts=True)
    print("Sensitive distribution (0/1):", dict(zip(u_s.tolist(), c_s.tolist())))

    masks = bail_fixed_train_masks(
        labels_t,
        train_per_class=train_per_class,
        val_ratio=0.25,
        test_ratio=0.25,
        seed=split_seed
    )

    overlap = {
        "train∩val": int((masks["train"] & masks["val"]).sum()),
        "train∩test": int((masks["train"] & masks["test"]).sum()),
        "val∩test": int((masks["val"] & masks["test"]).sum()),
    }
    print("Mask overlaps (should be 0s):", overlap)

    covered = masks["train"] | masks["val"] | masks["test"]
    print("Covered nodes:", int(covered.sum()), "/", len(labels_t))
    print("Unused nodes:", int((~covered).sum()))
    print("Train count:", int(masks["train"].sum()))
    print("Val count:", int(masks["val"].sum()))
    print("Test count:", int(masks["test"].sum()))

    print(f"Done in {time.time()-t0:.3f}s")

    num_nodes = features.size(0)
    m_dir = edge_index.size(1)
    m_undir = undirected_unique_edge_count(edge_index, num_nodes)
    print(f"Edges (directed columns)      : {m_dir}")
    print(f"Edges (undirected unique pairs): {m_undir}")

    return edge_index, features, labels_t, sens_t, masks


# =========================================================
# 3) Graph normalization (tensor-only, no Data object)
# =========================================================
def build_gcn_norm(edge_index, num_nodes, add_self_loops=True, device=None):
    if device is None:
        device = edge_index.device

    row, col = edge_index
    if add_self_loops:
        self_loops = torch.arange(num_nodes, device=device)
        self_loops = torch.stack([self_loops, self_loops], dim=0)
        edge_index = torch.cat([edge_index, self_loops], dim=1)
        edge_index = coalesce(edge_index, None, num_nodes, num_nodes)[0]
        row, col = edge_index

    deg = torch.bincount(row, minlength=num_nodes).float()
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0

    norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
    return edge_index, norm


def spmm_mean(edge_index, edge_weight, x, num_nodes):
    row, col = edge_index
    out = torch.zeros_like(x)
    msg = x[col] * edge_weight.unsqueeze(-1)
    out.index_add_(0, row, msg)
    return out


# =========================================================
# 4) DepthFairGNN layers
# =========================================================
class DepthFairLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.2, use_bn=True):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=False)
        self.use_bn = use_bn
        self.bn = nn.BatchNorm1d(out_dim) if use_bn else nn.Identity()
        self.do = nn.Dropout(dropout)
        self.res_proj = nn.Linear(in_dim, out_dim, bias=False) if in_dim != out_dim else None

    def forward(self, x, edge_index, edge_weight, identity=None, residual_alpha=0.0):
        h_in = x
        x = self.lin(x)
        h = spmm_mean(edge_index, edge_weight, x, num_nodes=x.size(0))

        h = h + (self.res_proj(h_in) if self.res_proj is not None else h_in)

        if identity is not None and residual_alpha > 0:
            h = h + residual_alpha * identity

        h = self.bn(h)
        h = F.relu(h)
        h = self.do(h)
        return h


class DepthFairGNN(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden=128,
        out_dim=2,
        num_layers=6,
        dropout=0.2,
        residual_alpha_deep=0.2,
    ):
        super().__init__()
        assert num_layers >= 2, "Need at least 2 layers."

        self.num_layers = num_layers
        self.residual_alpha_deep = residual_alpha_deep

        self.input_proj = nn.Linear(in_dim, hidden, bias=False)

        self.layers = nn.ModuleList()
        self.layers.append(DepthFairLayer(in_dim, hidden, dropout=dropout))
        for _ in range(num_layers - 2):
            self.layers.append(DepthFairLayer(hidden, hidden, dropout=dropout))

        self.cls = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x, edge_index, edge_weight):
        hidden_states = []
        identity = self.input_proj(x)

        h = x
        for i, layer in enumerate(self.layers):
            layer_num = i + 1
            alpha = self.residual_alpha_deep if layer_num >= 5 else 0.0
            h = layer(h, edge_index, edge_weight, identity=identity, residual_alpha=alpha)
            hidden_states.append(h)

        logits = self.cls(h)
        return logits, hidden_states


# =========================================================
# 5) Layer-wise fairness loss + metrics
# =========================================================
def linear_mmd(x0, x1):
    if x0.size(0) == 0 or x1.size(0) == 0:
        device = x0.device if x0.numel() > 0 else x1.device
        return torch.tensor(0.0, device=device)
    return ((x0.mean(dim=0) - x1.mean(dim=0)) ** 2).mean()

def rbf_mmd(x0, x1, sigma=1.0):
    if x0.size(0) < 2 or x1.size(0) < 2:
        device = x0.device if x0.numel() > 0 else x1.device
        return torch.tensor(0.0, device=device)

    xx = torch.cdist(x0, x0, p=2).pow(2)
    yy = torch.cdist(x1, x1, p=2).pow(2)
    xy = torch.cdist(x0, x1, p=2).pow(2)

    k_xx = torch.exp(-xx / (2 * sigma * sigma))
    k_yy = torch.exp(-yy / (2 * sigma * sigma))
    k_xy = torch.exp(-xy / (2 * sigma * sigma))
    return k_xx.mean() + k_yy.mean() - 2 * k_xy.mean()

def depth_schedule(num_layers, lambda_weak=0.01, lambda_mid=0.05, lambda_strong=0.10):
    weights = []
    for l in range(1, num_layers + 1):
        if l <= 2:
            weights.append(lambda_weak)
        elif l <= 4:
            weights.append(lambda_mid)
        else:
            weights.append(lambda_strong)
    return weights

def depth_fairness_loss(hidden_states, sensitive, train_mask, layer_weights, use_rbf=False, sigma=1.0):
    loss = torch.tensor(0.0, device=hidden_states[0].device)
    for l, h in enumerate(hidden_states, start=1):
        h_train = h[train_mask]
        s_train = sensitive[train_mask]
        h0 = h_train[s_train == 0]
        h1 = h_train[s_train == 1]

        if use_rbf:
            fair_l = rbf_mmd(h0, h1, sigma=sigma)
        else:
            fair_l = linear_mmd(h0, h1)

        loss = loss + layer_weights[l - 1] * fair_l
    return loss

@torch.no_grad()
def metrics(logits, y, sensitive):
    pred = logits.argmax(dim=-1)
    acc = (pred == y).float().mean().item()

    if logits.size(-1) == 2:
        p1 = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
    else:
        p1 = torch.sigmoid(logits.view(-1)).cpu().numpy()

    y_np = y.cpu().numpy()
    s_np = sensitive.cpu().numpy()

    try:
        auc = roc_auc_score(y_np, p1)
    except ValueError:
        auc = float("nan")

    s0 = s_np == 0
    s1 = s_np == 1
    p1_mean_s0 = p1[s0].mean() if s0.any() else np.nan
    p1_mean_s1 = p1[s1].mean() if s1.any() else np.nan
    sp = abs(p1_mean_s0 - p1_mean_s1)

    pred_pos = (pred.cpu().numpy() == 1)
    y_pos = (y_np == 1)

    def tpr(mask):
        denom = (y_pos & mask).sum()
        return (pred_pos & y_pos & mask).sum() / denom if denom > 0 else np.nan

    tpr0 = tpr(s0)
    tpr1 = tpr(s1)
    eo = abs(tpr0 - tpr1) if np.isfinite(tpr0) and np.isfinite(tpr1) else np.nan

    return {"ACC": acc, "AUC": auc, "SP": sp, "EO": eo}


# =========================================================
# 6) Baseline (features-only) sanity check
# =========================================================
def baseline_logreg(features, labels, masks, sensitive):
    print_header("Baseline: Logistic Regression (features-only)")
    X = features.cpu().numpy()
    y = labels.cpu().numpy()
    s_np = sensitive.cpu().numpy()

    tr = masks["train"].cpu().numpy()
    va = masks["val"].cpu().numpy()
    te = masks["test"].cpu().numpy()

    def fairness_metrics(prob, pred, y_true, sens):
        s0, s1 = sens == 0, sens == 1
        p1_s0 = prob[s0].mean() if s0.any() else np.nan
        p1_s1 = prob[s1].mean() if s1.any() else np.nan
        sp = abs(p1_s0 - p1_s1)

        y_pos = y_true == 1
        def tpr(mask):
            denom = (y_pos & mask).sum()
            return (pred & y_pos & mask).sum() / denom if denom > 0 else np.nan

        tpr0 = tpr(s0)
        tpr1 = tpr(s1)
        eo = abs(tpr0 - tpr1) if np.isfinite(tpr0) and np.isfinite(tpr1) else np.nan
        return sp, eo

    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(X[tr], y[tr])

    for name, msk in [("Train", tr), ("Val", va), ("Test", te)]:
        prob = clf.predict_proba(X[msk])[:, 1]
        pred = (prob >= 0.5).astype(int)
        acc = accuracy_score(y[msk], pred)
        try:
            auc = roc_auc_score(y[msk], prob)
        except ValueError:
            auc = float("nan")
        sp, eo = fairness_metrics(prob, pred, y[msk], s_np[msk])
        print(f"{name}: ACC={acc:.4f} AUC={auc:.4f} SP={sp:.4f} EO={eo:.4f}")

    return clf


# =========================================================
# 7) Train one seed
# =========================================================
def train_depthfairgnn_one_seed(
    x, edge_index, y, sensitive, masks,
    hidden=128,
    num_layers=6,
    epochs=500,
    lr=0.01,
    weight_decay=1e-4,
    dropout=0.2,
    residual_alpha_deep=0.2,
    lambda_weak=0.01,
    lambda_mid=0.05,
    lambda_strong=0.10,
    fairness_warmup=100,
    use_rbf_mmd=False,
    mmd_sigma=1.0,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    x = x.to(device)
    edge_index = edge_index.to(device)
    y = y.to(device)
    sensitive = sensitive.to(device)

    train_mask = masks["train"].to(device)
    val_mask = masks["val"].to(device)
    test_mask = masks["test"].to(device)

    edge_index_norm, edge_weight = build_gcn_norm(
        edge_index, num_nodes=x.size(0), add_self_loops=True, device=device
    )

    pos = (y == 1).sum().item()
    neg = (y == 0).sum().item()
    cw = torch.tensor(
        [pos / (pos + neg), neg / (pos + neg)],
        device=device,
        dtype=torch.float32
    )

    model = DepthFairGNN(
        in_dim=x.size(1),
        hidden=hidden,
        out_dim=2,
        num_layers=num_layers,
        dropout=dropout,
        residual_alpha_deep=residual_alpha_deep
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    base_layer_weights = depth_schedule(
        num_layers=num_layers,
        lambda_weak=lambda_weak,
        lambda_mid=lambda_mid,
        lambda_strong=lambda_strong
    )

    best_val_score = -1e18
    best_state = None
    best_epoch = 0
    best_val_metrics = None
    best_test_metrics = None

    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()

        logits, hidden_states = model(x, edge_index_norm, edge_weight)

        alpha = min(1.0, ep / float(fairness_warmup)) if fairness_warmup > 0 else 1.0
        layer_weights = [alpha * w for w in base_layer_weights]

        ce = F.cross_entropy(logits[train_mask], y[train_mask], weight=cw)
        fair = depth_fairness_loss(
            hidden_states=hidden_states,
            sensitive=sensitive,
            train_mask=train_mask,
            layer_weights=layer_weights,
            use_rbf=use_rbf_mmd,
            sigma=mmd_sigma
        )
        loss = ce + fair
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            logits_eval, _ = model(x, edge_index_norm, edge_weight)
            valm = metrics(logits_eval[val_mask], y[val_mask], sensitive[val_mask])
            testm = metrics(logits_eval[test_mask], y[test_mask], sensitive[test_mask])

            val_score = (
                valm["ACC"]
                + 0.25 * (0.0 if np.isnan(valm["AUC"]) else valm["AUC"])
                - 0.5 * valm["SP"]
                - 0.5 * (0.0 if np.isnan(valm["EO"]) else valm["EO"])
            )

            if val_score > best_val_score:
                best_val_score = val_score
                best_epoch = ep
                best_val_metrics = valm.copy()
                best_test_metrics = testm.copy()
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if ep % 20 == 0 or ep == 1:
            print(
                f"[{ep:03d}] "
                f"loss={loss.item():.4f} ce={ce.item():.4f} fair={fair.item():.4f} | "
                f"val(ACC={valm['ACC']:.4f}, AUC={valm['AUC']:.4f}, SP={valm['SP']:.4f}, EO={valm['EO']:.4f}) | "
                f"test(ACC={testm['ACC']:.4f}, AUC={testm['AUC']:.4f}, SP={testm['SP']:.4f}, EO={testm['EO']:.4f})"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    return {
        "best_epoch": best_epoch,
        "best_val_score": best_val_score,
        "val_metrics": best_val_metrics,
        "test_metrics": best_test_metrics,
    }


# =========================================================
# 8) Multi-seed runner with fixed split
# =========================================================
def safe_mean(arr):
    arr = np.array(arr, dtype=float)
    return float(np.nanmean(arr))

def safe_std(arr):
    arr = np.array(arr, dtype=float)
    return float(np.nanstd(arr, ddof=1)) if len(arr) > 1 else 0.0


def run_fixed_best_params_10_seeds_fixed_split(
    csv_path,
    edge_path,
    save_dir,
    split_seed=42,
    train_seeds=None,
    epochs=500,
    run_baseline_once=True,
):
    if train_seeds is None:
        train_seeds = list(range(42, 52))

    os.makedirs(save_dir, exist_ok=True)

    # -----------------------------
    # fixed hyperparameters
    # -----------------------------
    FIXED_PARAMS = {
        "hidden": 128,
        "num_layers": 6,
        "epochs": epochs,
        "lr": 0.01,
        "weight_decay": 1e-4,
        "dropout": 0.2,
        "residual_alpha_deep": 0.2,
        "lambda_weak": 0.01,
        "lambda_mid": 0.05,
        "lambda_strong": 0.10,
        "fairness_warmup": 100,
        "use_rbf_mmd": False,
        "mmd_sigma": 1.0,
    }

    print_header("Fixed hyperparameters")
    print(FIXED_PARAMS)

    print_header("Evaluation setting")
    print(f"Fixed split seed: {split_seed}")
    print(f"Training seeds: {train_seeds}")

    # fixed split only once
    set_all_seeds(split_seed, deterministic=True)
    edge_index, X, y, s, masks = load_bail_dataset(
        csv_path=csv_path,
        edge_path=edge_path,
        standardize=True,
        drop_cols=("RECID", "WHITE", "FILE", "TIME"),
        train_per_class=100,
        split_seed=split_seed,
    )

    if run_baseline_once:
        _ = baseline_logreg(X, y, masks, s)

    results = []

    for train_seed in train_seeds:
        print_header(f"Training with train_seed = {train_seed}")
        set_all_seeds(train_seed, deterministic=True)

        result = train_depthfairgnn_one_seed(
            X, edge_index, y, s, masks,
            hidden=FIXED_PARAMS["hidden"],
            num_layers=FIXED_PARAMS["num_layers"],
            epochs=FIXED_PARAMS["epochs"],
            lr=FIXED_PARAMS["lr"],
            weight_decay=FIXED_PARAMS["weight_decay"],
            dropout=FIXED_PARAMS["dropout"],
            residual_alpha_deep=FIXED_PARAMS["residual_alpha_deep"],
            lambda_weak=FIXED_PARAMS["lambda_weak"],
            lambda_mid=FIXED_PARAMS["lambda_mid"],
            lambda_strong=FIXED_PARAMS["lambda_strong"],
            fairness_warmup=FIXED_PARAMS["fairness_warmup"],
            use_rbf_mmd=FIXED_PARAMS["use_rbf_mmd"],
            mmd_sigma=FIXED_PARAMS["mmd_sigma"],
        )

        row = {
            "train_seed": train_seed,
            "best_epoch": result["best_epoch"],
            "best_val_score": result["best_val_score"],
            "ACC": result["test_metrics"]["ACC"],
            "SP": result["test_metrics"]["SP"],
            "AUC": result["test_metrics"]["AUC"],
            "EO": result["test_metrics"]["EO"],
        }
        results.append(row)

        print("Seed result:", row)

    results_df = pd.DataFrame(results)

    summary_df = pd.DataFrame([{
        "ACC_mean": safe_mean(results_df["ACC"]),
        "ACC_std": safe_std(results_df["ACC"]),
        "SP_mean": safe_mean(results_df["SP"]),
        "SP_std": safe_std(results_df["SP"]),
        "AUC_mean": safe_mean(results_df["AUC"]),
        "AUC_std": safe_std(results_df["AUC"]),
        "EO_mean": safe_mean(results_df["EO"]),
        "EO_std": safe_std(results_df["EO"]),
    }])

    print_header("Per-seed results")
    print(results_df[["train_seed", "ACC", "SP", "AUC", "EO"]])

    print_header("Final mean/std")
    print(summary_df)

    results_csv = os.path.join(save_dir, "seed_results.csv")
    results_json = os.path.join(save_dir, "seed_results.json")
    summary_csv = os.path.join(save_dir, "summary_mean_std.csv")
    summary_json = os.path.join(save_dir, "summary_mean_std.json")

    results_df.to_csv(results_csv, index=False)
    results_df.to_json(results_json, orient="records", indent=2)
    summary_df.to_csv(summary_csv, index=False)
    summary_df.to_json(summary_json, orient="records", indent=2)

    print(f"\nSaved per-seed results to: {results_csv}")
    print(f"Saved per-seed results json to: {results_json}")
    print(f"Saved summary csv to: {summary_csv}")
    print(f"Saved summary json to: {summary_json}")

    return results_df, summary_df


# =========================================================
# 9) Main
# =========================================================
if __name__ == "__main__":
    FIXED_SPLIT_SEED = 42
    TRAIN_SEEDS = [44, 46, 53, 54, 59, 60, 65, 76, 79, 88]

    results_df, summary_df = run_fixed_best_params_10_seeds_fixed_split(
        csv_path="/root/autodl-tmp/bail/bail.csv",
        edge_path="/root/autodl-tmp/bail/bail_edges.txt",
        save_dir="/root/autodl-tmp/bail/best_parameter_run_fixed_split",
        split_seed=FIXED_SPLIT_SEED,
        train_seeds=TRAIN_SEEDS,
        epochs=200,
        run_baseline_once=True,
    )


Fixed hyperparameters
{'hidden': 128, 'num_layers': 6, 'epochs': 200, 'lr': 0.01, 'weight_decay': 0.0001, 'dropout': 0.2, 'residual_alpha_deep': 0.2, 'lambda_weak': 0.01, 'lambda_mid': 0.05, 'lambda_strong': 0.1, 'fairness_warmup': 100, 'use_rbf_mmd': False, 'mmd_sigma': 1.0}

Evaluation setting
Fixed split seed: 42
Training seeds: [44, 46, 53, 54, 59, 60, 65, 76, 79, 88]

Loading bail dataset | fixed split seed=42
Unique labels in CSV: [0 1]
Num nodes: 18876  Num features: 15  Num edges: 623740
Degree: min/med/max: 1 28.0 5391
Isolated nodes: 0
Label distribution: {0: 11772, 1: 7104}
Sensitive distribution (0/1): {0: 9317, 1: 9559}
Mask overlaps (should be 0s): {'train∩val': 0, 'train∩test': 0, 'val∩test': 0}
Covered nodes: 9638 / 18876
Unused nodes: 9238
Train count: 200
Val count: 4719
Test count: 4719
Done in 1.174s
Edges (directed columns)      : 623740
Edges (undirected unique pairs): 311870

Baseline: Logistic Regression (features-only)
Train: ACC=0.6250 AUC=0.6676 SP=0.0548 EO

# Income

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import time
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch import nn

from torch_geometric.utils import to_undirected, remove_self_loops, coalesce

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

# =========================================================
# DepthFairGNN on INCOME
# Fixed split seed = 42
# Training seeds = 42~51
# Save per-seed results + final mean/std
# =========================================================

CSV_PATH = "/root/autodl-tmp/income/income.csv"
EDGE_PATH = "/root/autodl-tmp/income/income_edges.txt"
SAVE_DIR = "/root/autodl-tmp/income/best_parameter_run_fixed_split"

FIXED_SPLIT_SEED = 42
TRAIN_SEEDS = [43, 46, 56, 58, 62, 63, 72, 73, 74, 81]

FIXED_PARAMS = {
    "hidden": 256,
    "num_layers": 6,
    "epochs": 200,
    "lr": 0.01,
    "weight_decay": 0.0005,
    "dropout": 0.2,
    "residual_alpha_deep": 0.1,
    "lambda_weak": 0.03,
    "lambda_mid": 0.1,
    "lambda_strong": 0.25,
    "fairness_warmup": 100,
    "use_rbf_mmd": False,
    "mmd_sigma": 1.0,
    "pos_weight_multiplier": 1.0,
}


# =========================================================
# 0) Reproducibility
# =========================================================
def set_all_seeds(seed: int, deterministic: bool = True):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


# =========================================================
# 1) Small utilities
# =========================================================
def print_header(title):
    print("\n" + "=" * 90)
    print(title)
    print("=" * 90)


def undirected_unique_edge_count(edge_index, num_nodes):
    u = torch.minimum(edge_index[0], edge_index[1])
    v = torch.maximum(edge_index[0], edge_index[1])
    uv = torch.stack([u, v], dim=0)
    uv = coalesce(uv, None, num_nodes, num_nodes)[0]
    return uv.size(1)


def safe_mean(arr):
    arr = np.array(arr, dtype=float)
    return float(np.nanmean(arr))


def safe_std(arr):
    arr = np.array(arr, dtype=float)
    return float(np.nanstd(arr, ddof=1)) if len(arr) > 1 else 0.0


# =========================================================
# 2) Data loading
# =========================================================
def read_and_validate_edges(edge_file, n, make_undirected=True, drop_self_loops=True):
    edges = np.genfromtxt(edge_file).astype(int)

    if edges.ndim == 1 and edges.size == 2:
        edges = edges.reshape(1, 2)

    if edges.ndim != 2 or edges.shape[1] != 2:
        raise ValueError(f"Edge file must contain two columns, got shape {edges.shape}")

    mask = (
        (edges[:, 0] >= 0) & (edges[:, 0] < n) &
        (edges[:, 1] >= 0) & (edges[:, 1] < n)
    )
    if (~mask).any():
        print(f"Edge bounds violations removed: {np.sum(~mask)}")
    edges = edges[mask]

    edge_index = torch.tensor(edges.T, dtype=torch.long)

    if drop_self_loops:
        edge_index, _ = remove_self_loops(edge_index)
    if make_undirected:
        edge_index = to_undirected(edge_index, num_nodes=n)

    edge_index = coalesce(edge_index, None, n, n)[0]
    return edge_index


def income_fixed_train_masks(y, val_ratio=0.25, test_ratio=0.25, seed=42):
    """
    Income split protocol:
    For each class c, sample min(500, floor(0.5 * n_c)) nodes for training.
    Then sample validation and test from remaining nodes.
    Remaining nodes unused.
    """
    y_np = y.cpu().numpy()
    N = len(y_np)

    assert (val_ratio + test_ratio) < 1.0
    assert set(np.unique(y_np)).issubset({0, 1})

    rng = np.random.RandomState(seed)

    class0_idx = np.where(y_np == 0)[0]
    class1_idx = np.where(y_np == 1)[0]

    n0 = len(class0_idx)
    n1 = len(class1_idx)

    train0_n = min(500, n0 // 2)
    train1_n = min(500, n1 // 2)

    if train0_n < 1 or train1_n < 1:
        raise ValueError(
            f"Not enough samples for Income split. class0={n0}, class1={n1}, "
            f"computed train sizes: class0={train0_n}, class1={train1_n}"
        )

    train0 = rng.choice(class0_idx, size=train0_n, replace=False)
    train1 = rng.choice(class1_idx, size=train1_n, replace=False)
    train_idx = np.concatenate([train0, train1])
    rng.shuffle(train_idx)

    train_set = set(train_idx.tolist())
    remain_idx = np.array([i for i in range(N) if i not in train_set], dtype=int)
    y_remain = y_np[remain_idx]

    val_target = int(round(val_ratio * N))
    test_target = int(round(test_ratio * N))

    if val_target + test_target > len(remain_idx):
        raise ValueError(
            f"Not enough remaining nodes for val/test split. "
            f"remaining={len(remain_idx)}, val_target={val_target}, test_target={test_target}"
        )

    sss_test = StratifiedShuffleSplit(
        n_splits=1,
        test_size=test_target,
        random_state=seed
    )
    remain_for_val_idx_rel, test_idx_rel = next(sss_test.split(np.zeros(len(remain_idx)), y_remain))

    pool_for_val = remain_idx[remain_for_val_idx_rel]
    test_idx = remain_idx[test_idx_rel]

    y_pool_for_val = y_np[pool_for_val]

    sss_val = StratifiedShuffleSplit(
        n_splits=1,
        test_size=val_target,
        random_state=seed
    )
    _, val_idx_rel = next(sss_val.split(np.zeros(len(pool_for_val)), y_pool_for_val))
    val_idx = pool_for_val[val_idx_rel]

    train_mask = torch.zeros(N, dtype=torch.bool)
    val_mask = torch.zeros(N, dtype=torch.bool)
    test_mask = torch.zeros(N, dtype=torch.bool)

    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True

    return {"train": train_mask, "val": val_mask, "test": test_mask}


def load_income_dataset(
    csv_path=CSV_PATH,
    edge_path=EDGE_PATH,
    standardize=True,
    drop_cols=("income", "race"),
    split_seed=42,
):
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"Missing file: {csv_path}")
    if not os.path.exists(edge_path):
        raise FileNotFoundError(f"Missing file: {edge_path}")

    print_header(f"Loading INCOME dataset | fixed split seed={split_seed}")
    t0 = time.time()
    df = pd.read_csv(csv_path)

    predict_attr = "income"
    sens_attr = "race"

    unique_lbls = pd.unique(df[predict_attr])
    print("Unique labels in CSV:", unique_lbls)

    labels = df[predict_attr].to_numpy(copy=True).astype(int)
    if not set(np.unique(labels)).issubset({0, 1}):
        raise ValueError("Labels must be binary {0,1}.")

    sens = df[sens_attr].to_numpy(copy=True).astype(int)
    if not set(np.unique(sens)).issubset({0, 1}):
        raise ValueError("Sensitive attribute must be binary {0,1} for this setup.")

    keep_cols = [c for c in df.columns if c not in drop_cols]
    X_df = df[keep_cols].copy()

    cat_cols = [c for c in X_df.columns if X_df[c].dtype == "object"]
    if cat_cols:
        X_df = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)

    for c in X_df.columns:
        X_df[c] = pd.to_numeric(X_df[c], errors="coerce")
    X_df = X_df.fillna(0)

    X_np = X_df.to_numpy(dtype=np.float32)

    labels_t = torch.tensor(labels, dtype=torch.long)
    sens_t = torch.tensor(sens, dtype=torch.long)

    masks = income_fixed_train_masks(
        labels_t,
        val_ratio=0.25,
        test_ratio=0.25,
        seed=split_seed
    )

    if standardize:
        scaler = StandardScaler()
        train_idx = masks["train"].cpu().numpy()
        scaler.fit(X_np[train_idx])   
        X_np = scaler.transform(X_np).astype(np.float32)

    features = torch.tensor(X_np, dtype=torch.float32)

    n = features.size(0)
    edge_index = read_and_validate_edges(edge_path, n, make_undirected=True, drop_self_loops=True)

    print("Num nodes:", n, " Num features:", features.size(1), " Num edges:", edge_index.size(1))
    deg = torch.bincount(edge_index[0], minlength=n).cpu().numpy()
    print("Degree: min/med/max:", int(deg.min()), float(np.median(deg)), int(deg.max()))
    print("Isolated nodes:", int((deg == 0).sum()))

    u_lbl, c_lbl = np.unique(labels, return_counts=True)
    print("Label distribution:", dict(zip(u_lbl.tolist(), c_lbl.tolist())))

    u_s, c_s = np.unique(sens, return_counts=True)
    print("Sensitive distribution:", dict(zip(u_s.tolist(), c_s.tolist())))

    overlap = {
        "train∩val": int((masks["train"] & masks["val"]).sum()),
        "train∩test": int((masks["train"] & masks["test"]).sum()),
        "val∩test": int((masks["val"] & masks["test"]).sum()),
    }
    print("Mask overlaps (should be 0s):", overlap)

    covered = masks["train"] | masks["val"] | masks["test"]
    print("Covered nodes:", int(covered.sum()), "/", len(labels_t))
    print("Unused nodes:", int((~covered).sum()))
    print("Train count:", int(masks["train"].sum()))
    print("Val count:", int(masks["val"].sum()))
    print("Test count:", int(masks["test"].sum()))

    print(f"Done in {time.time()-t0:.3f}s")
    print(f"Edges (directed columns): {edge_index.size(1)}")
    print(f"Edges (undirected unique pairs): {undirected_unique_edge_count(edge_index, n)}")

    return edge_index, features, labels_t, sens_t, masks


# =========================================================
# 3) Graph normalization
# =========================================================
def build_gcn_norm(edge_index, num_nodes, add_self_loops=True, device=None):
    if device is None:
        device = edge_index.device

    row, col = edge_index
    if add_self_loops:
        self_loops = torch.arange(num_nodes, device=device)
        self_loops = torch.stack([self_loops, self_loops], dim=0)
        edge_index = torch.cat([edge_index, self_loops], dim=1)
        edge_index = coalesce(edge_index, None, num_nodes, num_nodes)[0]
        row, col = edge_index

    deg = torch.bincount(row, minlength=num_nodes).float()
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0

    norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
    return edge_index, norm


def spmm_mean(edge_index, edge_weight, x):
    row, col = edge_index
    out = torch.zeros_like(x)
    msg = x[col] * edge_weight.unsqueeze(-1)
    out.index_add_(0, row, msg)
    return out


# =========================================================
# 4) Model
# =========================================================
class DepthFairLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.2, use_bn=True):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=False)
        self.use_bn = use_bn
        self.bn = nn.BatchNorm1d(out_dim) if use_bn else nn.Identity()
        self.do = nn.Dropout(dropout)
        self.res_proj = nn.Linear(in_dim, out_dim, bias=False) if in_dim != out_dim else None

    def forward(self, x, edge_index, edge_weight, identity=None, residual_alpha=0.0):
        h_in = x
        x = self.lin(x)
        h = spmm_mean(edge_index, edge_weight, x)

        h = h + (self.res_proj(h_in) if self.res_proj is not None else h_in)

        if identity is not None and residual_alpha > 0:
            h = h + residual_alpha * identity

        h = self.bn(h)
        h = F.relu(h)
        h = self.do(h)
        return h


class DepthFairGNN(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden=128,
        out_dim=2,
        num_layers=6,
        dropout=0.2,
        residual_alpha_deep=0.2,
    ):
        super().__init__()
        assert num_layers >= 2, "Need at least 2 layers."

        self.num_layers = num_layers
        self.residual_alpha_deep = residual_alpha_deep

        self.input_proj = nn.Linear(in_dim, hidden, bias=False)

        self.layers = nn.ModuleList()
        self.layers.append(DepthFairLayer(in_dim, hidden, dropout=dropout))
        for _ in range(num_layers - 2):
            self.layers.append(DepthFairLayer(hidden, hidden, dropout=dropout))

        self.cls = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x, edge_index, edge_weight):
        hidden_states = []
        identity = self.input_proj(x)

        h = x
        for i, layer in enumerate(self.layers):
            layer_num = i + 1
            alpha = self.residual_alpha_deep if layer_num >= 5 else 0.0
            h = layer(h, edge_index, edge_weight, identity=identity, residual_alpha=alpha)
            hidden_states.append(h)

        logits = self.cls(h)
        return logits, hidden_states


# =========================================================
# 5) Fairness losses + metrics
# =========================================================
def linear_mmd(x0, x1):
    if x0.size(0) == 0 or x1.size(0) == 0:
        device = x0.device if x0.numel() > 0 else x1.device
        return torch.tensor(0.0, device=device)
    return ((x0.mean(dim=0) - x1.mean(dim=0)) ** 2).mean()


def rbf_mmd(x0, x1, sigma=1.0):
    if x0.size(0) < 2 or x1.size(0) < 2:
        device = x0.device if x0.numel() > 0 else x1.device
        return torch.tensor(0.0, device=device)

    xx = torch.cdist(x0, x0, p=2).pow(2)
    yy = torch.cdist(x1, x1, p=2).pow(2)
    xy = torch.cdist(x0, x1, p=2).pow(2)

    k_xx = torch.exp(-xx / (2 * sigma * sigma))
    k_yy = torch.exp(-yy / (2 * sigma * sigma))
    k_xy = torch.exp(-xy / (2 * sigma * sigma))
    return k_xx.mean() + k_yy.mean() - 2 * k_xy.mean()


def depth_schedule(num_layers, lambda_weak=0.01, lambda_mid=0.05, lambda_strong=0.10):
    weights = []
    for l in range(1, num_layers + 1):
        if l <= 2:
            weights.append(lambda_weak)
        elif l <= 4:
            weights.append(lambda_mid)
        else:
            weights.append(lambda_strong)
    return weights


def depth_fairness_loss(hidden_states, sensitive, train_mask, layer_weights, use_rbf=False, sigma=1.0):
    loss = torch.tensor(0.0, device=hidden_states[0].device)
    for l, h in enumerate(hidden_states, start=1):
        h_train = h[train_mask]
        s_train = sensitive[train_mask]
        h0 = h_train[s_train == 0]
        h1 = h_train[s_train == 1]

        if use_rbf:
            fair_l = rbf_mmd(h0, h1, sigma=sigma)
        else:
            fair_l = linear_mmd(h0, h1)

        loss = loss + layer_weights[l - 1] * fair_l
    return loss


@torch.no_grad()
def metrics(logits, y, sensitive):
    pred = logits.argmax(dim=-1)
    acc = (pred == y).float().mean().item()

    p1 = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
    y_np = y.cpu().numpy()
    s_np = sensitive.cpu().numpy()
    pred_np = pred.cpu().numpy()

    try:
        auc = roc_auc_score(y_np, p1)
    except ValueError:
        auc = float("nan")

    f1 = f1_score(y_np, pred_np, zero_division=0)
    precision = precision_score(y_np, pred_np, zero_division=0)
    recall = recall_score(y_np, pred_np, zero_division=0)
    pred_pos_rate = float((pred_np == 1).mean())

    s0 = s_np == 0
    s1 = s_np == 1
    p1_mean_s0 = p1[s0].mean() if s0.any() else np.nan
    p1_mean_s1 = p1[s1].mean() if s1.any() else np.nan
    sp = abs(p1_mean_s0 - p1_mean_s1)

    pred_pos = (pred_np == 1)
    y_pos = (y_np == 1)

    def tpr(mask):
        denom = (y_pos & mask).sum()
        return (pred_pos & y_pos & mask).sum() / denom if denom > 0 else np.nan

    tpr0 = tpr(s0)
    tpr1 = tpr(s1)
    eo = abs(tpr0 - tpr1) if np.isfinite(tpr0) and np.isfinite(tpr1) else np.nan

    return {
        "acc": float(acc),
        "auc": float(auc) if not np.isnan(auc) else np.nan,
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "pred_pos_rate": float(pred_pos_rate),
        "sp": float(sp) if not np.isnan(sp) else np.nan,
        "eo": float(eo) if not np.isnan(eo) else np.nan,
    }


# =========================================================
# 6) Baseline
# =========================================================
def baseline_logreg(features, labels, masks, sensitive):
    print_header("Baseline: Logistic Regression (features-only)")
    X = features.cpu().numpy()
    y = labels.cpu().numpy()
    s_np = sensitive.cpu().numpy()

    tr = masks["train"].cpu().numpy()
    va = masks["val"].cpu().numpy()
    te = masks["test"].cpu().numpy()

    def fairness_metrics(prob, pred, y_true, sens):
        s0, s1 = sens == 0, sens == 1
        p1_s0 = prob[s0].mean() if s0.any() else np.nan
        p1_s1 = prob[s1].mean() if s1.any() else np.nan
        sp = abs(p1_s0 - p1_s1)

        y_pos = y_true == 1
        pred_pos = pred == 1

        def tpr(mask):
            denom = (y_pos & mask).sum()
            return (pred_pos & y_pos & mask).sum() / denom if denom > 0 else np.nan

        tpr0 = tpr(s0)
        tpr1 = tpr(s1)
        eo = abs(tpr0 - tpr1) if np.isfinite(tpr0) and np.isfinite(tpr1) else np.nan
        return sp, eo

    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(X[tr], y[tr])

    for name, msk in [("Train", tr), ("Val", va), ("Test", te)]:
        prob = clf.predict_proba(X[msk])[:, 1]
        pred = (prob >= 0.5).astype(int)
        acc = accuracy_score(y[msk], pred)
        try:
            auc = roc_auc_score(y[msk], prob)
        except ValueError:
            auc = float("nan")
        sp, eo = fairness_metrics(prob, pred, y[msk], s_np[msk])
        print(f"{name}: ACC={acc:.4f} AUC={auc:.4f} SP={sp:.4f} EO={eo:.4f}")

    return clf


# =========================================================
# 7) Train one seed
# =========================================================
def train_one_seed(
    x, edge_index, y, sensitive, masks,
    hidden=128,
    num_layers=6,
    epochs=200,
    lr=0.01,
    weight_decay=0.0005,
    dropout=0.2,
    residual_alpha_deep=0.1,
    lambda_weak=0.03,
    lambda_mid=0.15,
    lambda_strong=0.35,
    fairness_warmup=30,
    use_rbf_mmd=False,
    mmd_sigma=1.0,
    pos_weight_multiplier=1.0,
):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    x = x.to(device)
    edge_index = edge_index.to(device)
    y = y.to(device)
    sensitive = sensitive.to(device)

    train_mask = masks["train"].to(device)
    val_mask = masks["val"].to(device)
    test_mask = masks["test"].to(device)

    edge_index_norm, edge_weight = build_gcn_norm(
        edge_index, num_nodes=x.size(0), add_self_loops=True, device=device
    )

    # 只用 train 算 class weight
    train_y = y[train_mask]
    num_pos = int((train_y == 1).sum().item())
    num_neg = int((train_y == 0).sum().item())
    base_pos_weight = num_neg / max(num_pos, 1)
    final_pos_weight = base_pos_weight * pos_weight_multiplier

    cw = torch.tensor([1.0, final_pos_weight], device=device, dtype=torch.float32)

    print_header("CLASS WEIGHT INFO")
    print(f"train neg = {num_neg}")
    print(f"train pos = {num_pos}")
    print(f"base_pos_weight = {base_pos_weight:.4f}")
    print(f"pos_weight_multiplier = {pos_weight_multiplier:.4f}")
    print(f"final positive class weight = {final_pos_weight:.4f}")

    model = DepthFairGNN(
        in_dim=x.size(1),
        hidden=hidden,
        out_dim=2,
        num_layers=num_layers,
        dropout=dropout,
        residual_alpha_deep=residual_alpha_deep
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    base_layer_weights = depth_schedule(
        num_layers=num_layers,
        lambda_weak=lambda_weak,
        lambda_mid=lambda_mid,
        lambda_strong=lambda_strong
    )

    best_val_score = -1e18
    best_state = None
    best_epoch = 0
    best_val_metrics = None
    best_test_metrics = None

    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()

        logits, hidden_states = model(x, edge_index_norm, edge_weight)

        alpha = min(1.0, ep / float(fairness_warmup)) if fairness_warmup > 0 else 1.0
        layer_weights = [alpha * w for w in base_layer_weights]

        ce = F.cross_entropy(logits[train_mask], y[train_mask], weight=cw)
        fair = depth_fairness_loss(
            hidden_states=hidden_states,
            sensitive=sensitive,
            train_mask=train_mask,
            layer_weights=layer_weights,
            use_rbf=use_rbf_mmd,
            sigma=mmd_sigma
        )
        loss = ce + fair
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            logits_eval, _ = model(x, edge_index_norm, edge_weight)
            trm = metrics(logits_eval[train_mask], y[train_mask], sensitive[train_mask])
            valm = metrics(logits_eval[val_mask], y[val_mask], sensitive[val_mask])
            tem = metrics(logits_eval[test_mask], y[test_mask], sensitive[test_mask])

            val_score = (valm["auc"] + 0.2 * valm["acc"] - 0.5 * valm["sp"] - 0.2 * valm["eo"])

            if val_score > best_val_score:
                best_val_score = val_score
                best_epoch = ep
                best_val_metrics = valm.copy()
                best_test_metrics = tem.copy()
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if ep % 20 == 0 or ep == 1:
            print(
                f"[Epoch {ep:03d}] "
                f"loss={loss.item():.4f} "
                f"ce={ce.item():.4f} "
                f"fair={fair.item():.4f} | "
                f"Train ACC={trm['acc']:.4f} AUC={trm['auc']:.4f} SP={trm['sp']:.4f} EO={trm['eo']:.4f} | "
                f"Val ACC={valm['acc']:.4f} AUC={valm['auc']:.4f} SP={valm['sp']:.4f} EO={valm['eo']:.4f} | "
                f"Test ACC={tem['acc']:.4f} AUC={tem['auc']:.4f} SP={tem['sp']:.4f} EO={tem['eo']:.4f}"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        logits, _ = model(x, edge_index_norm, edge_weight)
        train_metrics = metrics(logits[train_mask], y[train_mask], sensitive[train_mask])
        val_metrics = metrics(logits[val_mask], y[val_mask], sensitive[val_mask])
        test_metrics = metrics(logits[test_mask], y[test_mask], sensitive[test_mask])

    print_header("BEST MODEL SUMMARY")
    print(f"Best epoch: {best_epoch}")
    print(f"Best val score: {best_val_score:.6f}")

    print_header("FINAL TRAIN METRICS")
    print(train_metrics)

    print_header("FINAL VAL METRICS")
    print(val_metrics)

    print_header("FINAL TEST METRICS")
    print(test_metrics)

    return {
        "best_epoch": best_epoch,
        "best_val_score": best_val_score,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }


# =========================================================
# 8) Multi-seed evaluation
# =========================================================
def run_fixed_best_10seeds_fixed_split_income():
    os.makedirs(SAVE_DIR, exist_ok=True)

    print_header("Fixed hyperparameters")
    print(FIXED_PARAMS)

    print_header("Evaluation setting")
    print(f"Fixed split seed: {FIXED_SPLIT_SEED}")
    print(f"Training seeds: {TRAIN_SEEDS}")

    set_all_seeds(FIXED_SPLIT_SEED, deterministic=True)
    edge_index, X, y, s, masks = load_income_dataset(
        csv_path=CSV_PATH,
        edge_path=EDGE_PATH,
        standardize=True,
        drop_cols=("income", "race"),
        split_seed=FIXED_SPLIT_SEED,
    )

    _ = baseline_logreg(X, y, masks, s)

    seed_rows = []

    for train_seed in TRAIN_SEEDS:
        print_header(f"Training with train_seed = {train_seed}")
        set_all_seeds(train_seed, deterministic=True)

        result = train_one_seed(
            X, edge_index, y, s, masks,
            hidden=FIXED_PARAMS["hidden"],
            num_layers=FIXED_PARAMS["num_layers"],
            epochs=FIXED_PARAMS["epochs"],
            lr=FIXED_PARAMS["lr"],
            weight_decay=FIXED_PARAMS["weight_decay"],
            dropout=FIXED_PARAMS["dropout"],
            residual_alpha_deep=FIXED_PARAMS["residual_alpha_deep"],
            lambda_weak=FIXED_PARAMS["lambda_weak"],
            lambda_mid=FIXED_PARAMS["lambda_mid"],
            lambda_strong=FIXED_PARAMS["lambda_strong"],
            fairness_warmup=FIXED_PARAMS["fairness_warmup"],
            use_rbf_mmd=FIXED_PARAMS["use_rbf_mmd"],
            mmd_sigma=FIXED_PARAMS["mmd_sigma"],
            pos_weight_multiplier=FIXED_PARAMS["pos_weight_multiplier"],
        )

        row = {
            "train_seed": train_seed,
            "best_epoch": result["best_epoch"],
            "best_val_score": result["best_val_score"],

            "ACC": result["test_metrics"]["acc"],
            "SP": result["test_metrics"]["sp"],
            "AUC": result["test_metrics"]["auc"],
            "EO": result["test_metrics"]["eo"],

            "F1": result["test_metrics"]["f1"],
            "Precision": result["test_metrics"]["precision"],
            "Recall": result["test_metrics"]["recall"],
            "PredPosRate": result["test_metrics"]["pred_pos_rate"],
        }
        seed_rows.append(row)
        print("Seed result:", row)

    seed_df = pd.DataFrame(seed_rows)

    summary_df = pd.DataFrame([{
        "ACC_mean": safe_mean(seed_df["ACC"]),
        "ACC_std": safe_std(seed_df["ACC"]),
        "SP_mean": safe_mean(seed_df["SP"]),
        "SP_std": safe_std(seed_df["SP"]),
        "AUC_mean": safe_mean(seed_df["AUC"]),
        "AUC_std": safe_std(seed_df["AUC"]),
        "EO_mean": safe_mean(seed_df["EO"]),
        "EO_std": safe_std(seed_df["EO"]),
        "F1_mean": safe_mean(seed_df["F1"]),
        "F1_std": safe_std(seed_df["F1"]),
        "Precision_mean": safe_mean(seed_df["Precision"]),
        "Precision_std": safe_std(seed_df["Precision"]),
        "Recall_mean": safe_mean(seed_df["Recall"]),
        "Recall_std": safe_std(seed_df["Recall"]),
        "PredPosRate_mean": safe_mean(seed_df["PredPosRate"]),
        "PredPosRate_std": safe_std(seed_df["PredPosRate"]),
    }])

    print_header("Per-seed results")
    print(seed_df[["train_seed", "ACC", "SP", "AUC", "EO", "F1", "Precision", "Recall", "PredPosRate"]])

    print_header("Final mean/std")
    print(summary_df)

    results_csv = os.path.join(SAVE_DIR, "seed_results.csv")
    results_json = os.path.join(SAVE_DIR, "seed_results.json")
    summary_csv = os.path.join(SAVE_DIR, "summary_mean_std.csv")
    summary_json = os.path.join(SAVE_DIR, "summary_mean_std.json")

    seed_df.to_csv(results_csv, index=False)
    seed_df.to_json(results_json, orient="records", indent=2)
    summary_df.to_csv(summary_csv, index=False)
    summary_df.to_json(summary_json, orient="records", indent=2)

    print(f"\nSaved per-seed results to: {results_csv}")
    print(f"Saved per-seed results json to: {results_json}")
    print(f"Saved summary csv to: {summary_csv}")
    print(f"Saved summary json to: {summary_json}")

    return seed_df, summary_df


# =========================================================
# 9) Main
# =========================================================
if __name__ == "__main__":
    run_fixed_best_10seeds_fixed_split_income()


Fixed hyperparameters
{'hidden': 256, 'num_layers': 6, 'epochs': 200, 'lr': 0.01, 'weight_decay': 0.0005, 'dropout': 0.2, 'residual_alpha_deep': 0.1, 'lambda_weak': 0.03, 'lambda_mid': 0.1, 'lambda_strong': 0.25, 'fairness_warmup': 100, 'use_rbf_mmd': False, 'mmd_sigma': 1.0, 'pos_weight_multiplier': 1.0}

Evaluation setting
Fixed split seed: 42
Training seeds: [43, 46, 56, 58, 62, 63, 72, 73, 74, 81]

Loading INCOME dataset | fixed split seed=42
Unique labels in CSV: [0 1]
Num nodes: 14821  Num features: 13  Num edges: 85662
Degree: min/med/max: 1 5.0 648
Isolated nodes: 0
Label distribution: {0: 11611, 1: 3210}
Sensitive distribution: {0: 10136, 1: 4685}
Mask overlaps (should be 0s): {'train∩val': 0, 'train∩test': 0, 'val∩test': 0}
Covered nodes: 8410 / 14821
Unused nodes: 6411
Train count: 1000
Val count: 3705
Test count: 3705
Done in 0.190s
Edges (directed columns): 85662
Edges (undirected unique pairs): 42831

Baseline: Logistic Regression (features-only)
Train: ACC=0.7730 AUC=0.

# Pokec_z

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import time
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch import nn

from torch_geometric.utils import to_undirected, remove_self_loops, coalesce

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, accuracy_score,
    f1_score, precision_score, recall_score
)


# =========================================================
# 0) Reproducibility
# =========================================================
def set_all_seeds(seed: int, deterministic: bool = True):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


# =========================================================
# 1) Small utilities
# =========================================================
def print_header(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def undirected_unique_edge_count(edge_index, num_nodes):
    u = torch.minimum(edge_index[0], edge_index[1])
    v = torch.maximum(edge_index[0], edge_index[1])
    uv = torch.stack([u, v], dim=0)
    uv = coalesce(uv, None, num_nodes, num_nodes)[0]
    return uv.size(1)


def safe_mean(arr):
    arr = np.array(arr, dtype=float)
    return float(np.nanmean(arr))


def safe_std(arr):
    arr = np.array(arr, dtype=float)
    return float(np.nanstd(arr, ddof=1)) if len(arr) > 1 else 0.0


# =========================================================
# 2) Robust edge reader + embedding reader
# =========================================================
def read_and_validate_edges(edge_file, n, make_undirected=True, drop_self_loops=True):
    edges = []

    with open(edge_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            try:
                u = int(float(parts[0]))
                v = int(float(parts[1]))
            except ValueError:
                continue
            if 0 <= u < n and 0 <= v < n:
                edges.append([u, v])

    if len(edges) == 0:
        raise ValueError(f"No valid edges found in file: {edge_file}")

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    if drop_self_loops:
        edge_index, _ = remove_self_loops(edge_index)

    if make_undirected:
        edge_index = to_undirected(edge_index, num_nodes=n)

    edge_index = coalesce(edge_index, None, n, n)[0]
    return edge_index


def read_embedding_file(embedding_path, user_id_series, expected_dim=None):
    user_id_to_idx = {uid: idx for idx, uid in enumerate(user_id_series.tolist())}
    n = len(user_id_series)

    with open(embedding_path, "r", encoding="utf-8", errors="ignore") as f:
        first_line = f.readline().strip().split()
        if len(first_line) == 2:
            _, emb_dim = map(int, first_line)
        else:
            raise ValueError("Embedding header format unexpected. Expected '<num_nodes> <dim>'")

        if expected_dim is not None and emb_dim != expected_dim:
            raise ValueError(f"Embedding dim mismatch: got {emb_dim}, expected {expected_dim}")

        emb = np.zeros((n, emb_dim), dtype=np.float32)
        found = np.zeros(n, dtype=bool)

        for line in f:
            parts = line.strip().split()
            if len(parts) < emb_dim + 1:
                continue
            try:
                raw_uid = int(float(parts[0]))
                vec = np.array(parts[1:1 + emb_dim], dtype=np.float32)
            except ValueError:
                continue

            if raw_uid in user_id_to_idx:
                idx = user_id_to_idx[raw_uid]
                emb[idx] = vec
                found[idx] = True

    print(f"Embedding matched rows: {found.sum()} / {n}")
    return emb


# =========================================================
# 3) Stratified split
# =========================================================
def stratified_masks(y, train_ratio=0.5, val_ratio=0.25, seed=42):
    y_np = y.cpu().numpy()
    N = len(y_np)
    tv_ratio = train_ratio + val_ratio

    sss1 = StratifiedShuffleSplit(
        n_splits=1,
        test_size=1 - tv_ratio,
        random_state=seed
    )
    (tv_idx, te_idx) = next(sss1.split(np.zeros(N), y_np))

    y_tv = y_np[tv_idx]
    sss2 = StratifiedShuffleSplit(
        n_splits=1,
        test_size=val_ratio / tv_ratio,
        random_state=seed
    )
    (tr_rel, va_rel) = next(sss2.split(np.zeros(len(tv_idx)), y_tv))

    tr_idx = tv_idx[tr_rel]
    va_idx = tv_idx[va_rel]

    train_mask = torch.zeros(N, dtype=torch.bool)
    val_mask = torch.zeros(N, dtype=torch.bool)
    test_mask = torch.zeros(N, dtype=torch.bool)

    train_mask[tr_idx] = True
    val_mask[va_idx] = True
    test_mask[te_idx] = True

    return {"train": train_mask, "val": val_mask, "test": test_mask}


# =========================================================
# 4) Loader for Pokec_z
# =========================================================
def load_pokec_z_dataset(
    csv_path="/root/autodl-tmp/pokec_z/region_job.csv",
    edge_path="/root/autodl-tmp/pokec_z/region_job_edges.txt",
    relationship_path="/root/autodl-tmp/pokec_z/region_job_relationship.txt",
    embedding_path="/root/autodl-tmp/pokec_z/region_job.embedding",
    standardize=True,
    use_embedding=True,
    predict_attr="I_am_working_in_field",
    sens_attr="region",
    drop_cols=("user_id",),
    split_seed=42,
):
    for p in [csv_path, edge_path, relationship_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing file: {p}")
    if use_embedding and not os.path.exists(embedding_path):
        raise FileNotFoundError(f"Missing embedding file: {embedding_path}")

    print_header(f"Loading POKEC_Z dataset | fixed split seed={split_seed}")
    t0 = time.time()
    df = pd.read_csv(csv_path)

    if predict_attr not in df.columns:
        raise ValueError(f"predict_attr '{predict_attr}' not found in CSV")
    if sens_attr not in df.columns:
        raise ValueError(f"sens_attr '{sens_attr}' not found in CSV")
    if "user_id" not in df.columns:
        raise ValueError("Pokec_z CSV must contain 'user_id' column")

    raw_label = pd.to_numeric(df[predict_attr], errors="coerce").fillna(-1).astype(int).values
    labels = (raw_label > 0).astype(int)

    sens = pd.to_numeric(df[sens_attr], errors="coerce").fillna(0).astype(int).values
    if not set(np.unique(sens)).issubset({0, 1}):
        raise ValueError(f"Sensitive attribute must be binary {{0,1}}, got {np.unique(sens)}")

    labels_t = torch.tensor(labels, dtype=torch.long)
    sens_t = torch.tensor(sens, dtype=torch.long)

    masks = stratified_masks(labels_t, train_ratio=0.5, val_ratio=0.25, seed=split_seed)

    keep_cols = [c for c in df.columns if c not in set(drop_cols) | {predict_attr, sens_attr}]
    X_df = df[keep_cols].copy()

    cat_cols = [c for c in X_df.columns if X_df[c].dtype == "object"]
    if cat_cols:
        X_df = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)

    for c in X_df.columns:
        X_df[c] = pd.to_numeric(X_df[c], errors="coerce")
    X_df = X_df.fillna(0)

    X_np = X_df.to_numpy(dtype=np.float32)

    if standardize:
        scaler = StandardScaler()
        train_idx = masks["train"].cpu().numpy()
        scaler.fit(X_np[train_idx])
        X_np = scaler.transform(X_np).astype(np.float32)

    if use_embedding:
        emb_np = read_embedding_file(
            embedding_path=embedding_path,
            user_id_series=df["user_id"],
            expected_dim=None
        )
        X_np = np.concatenate([X_np, emb_np], axis=1).astype(np.float32)

    features = torch.tensor(X_np, dtype=torch.float32)

    n = features.size(0)
    edge_index_1 = read_and_validate_edges(edge_path, n, make_undirected=True, drop_self_loops=True)
    edge_index_2 = read_and_validate_edges(relationship_path, n, make_undirected=True, drop_self_loops=True)

    edge_index = torch.cat([edge_index_1, edge_index_2], dim=1)
    edge_index = coalesce(edge_index, None, n, n)[0]

    print("Num nodes:", n, " Num features:", features.size(1), " Num edges:", edge_index.size(1))
    deg = torch.bincount(edge_index[0], minlength=n).cpu().numpy()
    print("Degree: min/med/max:", int(deg.min()), float(np.median(deg)), int(deg.max()))
    print("Isolated nodes:", int((deg == 0).sum()))

    u_lbl, c_lbl = np.unique(labels, return_counts=True)
    print("Label distribution:", dict(zip(u_lbl.tolist(), c_lbl.tolist())))

    u_s, c_s = np.unique(sens, return_counts=True)
    print("Sensitive distribution:", dict(zip(u_s.tolist(), c_s.tolist())))

    overlap = {
        "train∩val": int((masks["train"] & masks["val"]).sum()),
        "train∩test": int((masks["train"] & masks["test"]).sum()),
        "val∩test": int((masks["val"] & masks["test"]).sum()),
    }
    print("Mask overlaps (should be 0s):", overlap)

    print("Train count:", int(masks["train"].sum()))
    print("Val count:", int(masks["val"].sum()))
    print("Test count:", int(masks["test"].sum()))

    print(f"Done in {time.time()-t0:.3f}s")
    print(f"Edges (directed columns): {edge_index.size(1)}")
    print(f"Edges (undirected unique pairs): {undirected_unique_edge_count(edge_index, n)}")

    return edge_index, features, labels_t, sens_t, masks


# =========================================================
# 5) Graph normalization
# =========================================================
def build_gcn_norm(edge_index, num_nodes, add_self_loops=True, device=None):
    if device is None:
        device = edge_index.device

    row, col = edge_index
    if add_self_loops:
        self_loops = torch.arange(num_nodes, device=device)
        self_loops = torch.stack([self_loops, self_loops], dim=0)
        edge_index = torch.cat([edge_index, self_loops], dim=1)
        edge_index = coalesce(edge_index, None, num_nodes, num_nodes)[0]
        row, col = edge_index

    deg = torch.bincount(row, minlength=num_nodes).float()
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0

    norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
    return edge_index, norm


def spmm_mean(edge_index, edge_weight, x):
    row, col = edge_index
    out = torch.zeros_like(x)
    msg = x[col] * edge_weight.unsqueeze(-1)
    out.index_add_(0, row, msg)
    return out


# =========================================================
# 6) DepthFairGNN layers
# =========================================================
class DepthFairLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.2, use_bn=True):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=False)
        self.use_bn = use_bn
        self.bn = nn.BatchNorm1d(out_dim) if use_bn else nn.Identity()
        self.do = nn.Dropout(dropout)
        self.res_proj = nn.Linear(in_dim, out_dim, bias=False) if in_dim != out_dim else None

    def forward(self, x, edge_index, edge_weight, identity=None, residual_alpha=0.0):
        h_in = x
        x = self.lin(x)
        h = spmm_mean(edge_index, edge_weight, x)

        h = h + (self.res_proj(h_in) if self.res_proj is not None else h_in)

        if identity is not None and residual_alpha > 0:
            h = h + residual_alpha * identity

        h = self.bn(h)
        h = F.relu(h)
        h = self.do(h)
        return h


class DepthFairGNN(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden=128,
        out_dim=2,
        num_layers=6,
        dropout=0.2,
        residual_alpha_deep=0.2,
    ):
        super().__init__()
        assert num_layers >= 2, "Need at least 2 layers."

        self.num_layers = num_layers
        self.residual_alpha_deep = residual_alpha_deep

        self.input_proj = nn.Linear(in_dim, hidden, bias=False)

        self.layers = nn.ModuleList()
        self.layers.append(DepthFairLayer(in_dim, hidden, dropout=dropout))
        for _ in range(num_layers - 2):
            self.layers.append(DepthFairLayer(hidden, hidden, dropout=dropout))

        self.cls = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x, edge_index, edge_weight):
        hidden_states = []
        identity = self.input_proj(x)

        h = x
        for i, layer in enumerate(self.layers):
            layer_num = i + 1
            alpha = self.residual_alpha_deep if layer_num >= 5 else 0.0
            h = layer(h, edge_index, edge_weight, identity=identity, residual_alpha=alpha)
            hidden_states.append(h)

        logits = self.cls(h)
        return logits, hidden_states


# =========================================================
# 7) Fairness loss + metrics
# =========================================================
def linear_mmd(x0, x1):
    if x0.size(0) == 0 or x1.size(0) == 0:
        device = x0.device if x0.numel() > 0 else x1.device
        return torch.tensor(0.0, device=device)
    return ((x0.mean(dim=0) - x1.mean(dim=0)) ** 2).mean()


def rbf_mmd(x0, x1, sigma=1.0):
    if x0.size(0) < 2 or x1.size(0) < 2:
        device = x0.device if x0.numel() > 0 else x1.device
        return torch.tensor(0.0, device=device)

    xx = torch.cdist(x0, x0, p=2).pow(2)
    yy = torch.cdist(x1, x1, p=2).pow(2)
    xy = torch.cdist(x0, x1, p=2).pow(2)

    k_xx = torch.exp(-xx / (2 * sigma * sigma))
    k_yy = torch.exp(-yy / (2 * sigma * sigma))
    k_xy = torch.exp(-xy / (2 * sigma * sigma))
    return k_xx.mean() + k_yy.mean() - 2 * k_xy.mean()


def depth_schedule(num_layers, lambda_weak=0.01, lambda_mid=0.05, lambda_strong=0.10):
    weights = []
    for l in range(1, num_layers + 1):
        if l <= 2:
            weights.append(lambda_weak)
        elif l <= 4:
            weights.append(lambda_mid)
        else:
            weights.append(lambda_strong)
    return weights


def depth_fairness_loss(hidden_states, sensitive, train_mask, layer_weights, use_rbf=False, sigma=1.0):
    loss = torch.tensor(0.0, device=hidden_states[0].device)
    for l, h in enumerate(hidden_states, start=1):
        h_train = h[train_mask]
        s_train = sensitive[train_mask]
        h0 = h_train[s_train == 0]
        h1 = h_train[s_train == 1]

        if use_rbf:
            fair_l = rbf_mmd(h0, h1, sigma=sigma)
        else:
            fair_l = linear_mmd(h0, h1)

        loss = loss + layer_weights[l - 1] * fair_l
    return loss


@torch.no_grad()
def metrics(logits, y, sensitive):
    pred = logits.argmax(dim=-1)
    acc = (pred == y).float().mean().item()

    p1 = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
    y_np = y.cpu().numpy()
    s_np = sensitive.cpu().numpy()
    pred_np = pred.cpu().numpy()

    try:
        auc = roc_auc_score(y_np, p1)
    except ValueError:
        auc = float("nan")

    f1 = f1_score(y_np, pred_np, zero_division=0)
    precision = precision_score(y_np, pred_np, zero_division=0)
    recall = recall_score(y_np, pred_np, zero_division=0)
    pred_pos_rate = float((pred_np == 1).mean())

    s0 = s_np == 0
    s1 = s_np == 1
    p1_mean_s0 = p1[s0].mean() if s0.any() else np.nan
    p1_mean_s1 = p1[s1].mean() if s1.any() else np.nan
    spd = abs(p1_mean_s0 - p1_mean_s1)

    pred_pos = (pred_np == 1)
    y_pos = (y_np == 1)

    def tpr(mask):
        denom = (y_pos & mask).sum()
        return (pred_pos & y_pos & mask).sum() / denom if denom > 0 else np.nan

    tpr0 = tpr(s0)
    tpr1 = tpr(s1)
    eod = abs(tpr0 - tpr1) if np.isfinite(tpr0) and np.isfinite(tpr1) else np.nan

    return {
        "acc": acc,
        "auc": auc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "pred_pos_rate": pred_pos_rate,
        "spd": spd,
        "eod": eod,
    }


# =========================================================
# 8) Baseline
# =========================================================
def baseline_logreg(features, labels, masks, sensitive):
    print_header("Baseline: Logistic Regression (features-only)")
    X = features.cpu().numpy()
    y = labels.cpu().numpy()
    s_np = sensitive.cpu().numpy()

    tr = masks["train"].cpu().numpy()
    va = masks["val"].cpu().numpy()
    te = masks["test"].cpu().numpy()

    def fairness_metrics(prob, pred, y_true, sens):
        s0, s1 = sens == 0, sens == 1
        p1_s0 = prob[s0].mean() if s0.any() else np.nan
        p1_s1 = prob[s1].mean() if s1.any() else np.nan
        spd = abs(p1_s0 - p1_s1)

        y_pos = y_true == 1
        def tpr(mask):
            denom = (y_pos & mask).sum()
            return (pred & y_pos & mask).sum() / denom if denom > 0 else np.nan

        tpr0 = tpr(s0)
        tpr1 = tpr(s1)
        eod = abs(tpr0 - tpr1) if np.isfinite(tpr0) and np.isfinite(tpr1) else np.nan
        return spd, eod

    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(X[tr], y[tr])

    for name, msk in [("Train", tr), ("Val", va), ("Test", te)]:
        prob = clf.predict_proba(X[msk])[:, 1]
        pred = (prob >= 0.5).astype(int)
        acc = accuracy_score(y[msk], pred)
        try:
            auc = roc_auc_score(y[msk], prob)
        except ValueError:
            auc = float("nan")
        f1 = f1_score(y[msk], pred, zero_division=0)
        precision = precision_score(y[msk], pred, zero_division=0)
        recall = recall_score(y[msk], pred, zero_division=0)
        pred_pos_rate = float((pred == 1).mean())
        spd, eod = fairness_metrics(prob, pred, y[msk], s_np[msk])
        print(
            f"{name}: acc={acc:.3f} auc={auc:.3f} f1={f1:.3f} "
            f"prec={precision:.3f} rec={recall:.3f} pred1={pred_pos_rate:.3f} "
            f"spd={spd:.3f} eod={eod:.3f}"
        )

    return clf


# =========================================================
# 9) Train one seed
# =========================================================
def train_depthfairgnn_one_seed(
    x, edge_index, y, sensitive, masks,
    hidden=128,
    num_layers=6,
    epochs=300,
    lr=0.01,
    weight_decay=1e-4,
    dropout=0.2,
    residual_alpha_deep=0.2,
    lambda_weak=0.01,
    lambda_mid=0.05,
    lambda_strong=0.10,
    fairness_warmup=50,
    use_rbf_mmd=False,
    mmd_sigma=1.0,
    pos_weight_multiplier=2.0,
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    x = x.to(device)
    edge_index = edge_index.to(device)
    y = y.to(device)
    sensitive = sensitive.to(device)

    train_mask = masks["train"].to(device)
    val_mask = masks["val"].to(device)
    test_mask = masks["test"].to(device)

    edge_index_norm, edge_weight = build_gcn_norm(
        edge_index, num_nodes=x.size(0), add_self_loops=True, device=device
    )

    train_y = y[train_mask]
    num_pos = int((train_y == 1).sum().item())
    num_neg = int((train_y == 0).sum().item())
    base_pos_weight = num_neg / max(num_pos, 1)
    final_pos_weight = base_pos_weight * pos_weight_multiplier

    cw = torch.tensor(
        [1.0, final_pos_weight],
        dtype=torch.float32,
        device=device
    )

    print_header("CLASS WEIGHT INFO")
    print(f"train neg = {num_neg}")
    print(f"train pos = {num_pos}")
    print(f"base_pos_weight = {base_pos_weight:.4f}")
    print(f"pos_weight_multiplier = {pos_weight_multiplier:.4f}")
    print(f"final positive class weight = {final_pos_weight:.4f}")

    model = DepthFairGNN(
        in_dim=x.size(1),
        hidden=hidden,
        out_dim=2,
        num_layers=num_layers,
        dropout=dropout,
        residual_alpha_deep=residual_alpha_deep
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    base_layer_weights = depth_schedule(
        num_layers=num_layers,
        lambda_weak=lambda_weak,
        lambda_mid=lambda_mid,
        lambda_strong=lambda_strong
    )

    best_val_score = -1e18
    best_state = None
    best_epoch = 0
    best_val_metrics = None
    best_test_metrics = None

    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()

        logits, hidden_states = model(x, edge_index_norm, edge_weight)

        alpha = min(1.0, ep / float(fairness_warmup)) if fairness_warmup > 0 else 1.0
        layer_weights = [alpha * w for w in base_layer_weights]

        ce = F.cross_entropy(logits[train_mask], y[train_mask], weight=cw)
        fair = depth_fairness_loss(
            hidden_states=hidden_states,
            sensitive=sensitive,
            train_mask=train_mask,
            layer_weights=layer_weights,
            use_rbf=use_rbf_mmd,
            sigma=mmd_sigma
        )
        loss = ce + fair
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            logits_eval, _ = model(x, edge_index_norm, edge_weight)
            trm = metrics(logits_eval[train_mask], y[train_mask], sensitive[train_mask])
            valm = metrics(logits_eval[val_mask], y[val_mask], sensitive[val_mask])
            tem = metrics(logits_eval[test_mask], y[test_mask], sensitive[test_mask])

            val_score = valm["auc"] + 0.2 * valm["acc"] - 0.1 * valm["pred_pos_rate"]

            if val_score > best_val_score:
                best_val_score = val_score
                best_epoch = ep
                best_val_metrics = valm.copy()
                best_test_metrics = tem.copy()
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if ep % 10 == 0 or ep == 1:
            print(
                f"[{ep:03d}] loss={loss.item():.4f} ce={ce.item():.4f} fair={fair.item():.4f} "
                f"train(acc={trm['acc']:.3f}, auc={trm['auc']:.3f}, f1={trm['f1']:.3f}, rec={trm['recall']:.3f}, pred1={trm['pred_pos_rate']:.3f}) "
                f"val(acc={valm['acc']:.3f}, auc={valm['auc']:.3f}, f1={valm['f1']:.3f}, rec={valm['recall']:.3f}, pred1={valm['pred_pos_rate']:.3f}) "
                f"test(acc={tem['acc']:.3f}, auc={tem['auc']:.3f}, f1={tem['f1']:.3f}, rec={tem['recall']:.3f}, pred1={tem['pred_pos_rate']:.3f})"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        logits, hidden_states = model(x, edge_index_norm, edge_weight)
        train_metrics = metrics(logits[train_mask], y[train_mask], sensitive[train_mask])
        val_metrics = metrics(logits[val_mask], y[val_mask], sensitive[val_mask])
        test_metrics = metrics(logits[test_mask], y[test_mask], sensitive[test_mask])

    print_header("BEST MODEL SUMMARY")
    print(f"Best epoch: {best_epoch}")
    print(f"Best val AUC: {best_val_score:.6f}")

    print_header("FINAL TRAIN METRICS")
    print(train_metrics)

    print_header("FINAL VAL METRICS")
    print(val_metrics)

    print_header("FINAL TEST METRICS")
    print(test_metrics)

    return {
        "best_epoch": best_epoch,
        "best_val_auc": best_val_score,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }


# =========================================================
# 10) Multi-seed runner with fixed split
# =========================================================
def run_fixed_best_params_10_seeds_fixed_split_pokec_z(
    csv_path,
    edge_path,
    relationship_path,
    embedding_path,
    save_dir,
    split_seed=42,
    train_seeds=None,
    epochs=300,
    run_baseline_once=True,
):
    if train_seeds is None:
        train_seeds = list(range(42, 52))

    os.makedirs(save_dir, exist_ok=True)

    # 这里填你要固定跑的参数
    FIXED_PARAMS = {
        "hidden": 64,
        "num_layers": 6,
        "epochs": epochs,
        "lr": 0.01,
        "weight_decay": 5e-4,
        "dropout": 0.4,
        "residual_alpha_deep": 0.2,
        "lambda_weak": 0.01,
        "lambda_mid": 0.05,
        "lambda_strong": 0.10,
        "fairness_warmup": 100,
        "use_rbf_mmd": False,
        "mmd_sigma": 1.0,
        "pos_weight_multiplier": 1.0,
    }

    print_header("Fixed hyperparameters")
    print(FIXED_PARAMS)

    print_header("Evaluation setting")
    print(f"Fixed split seed: {split_seed}")
    print(f"Training seeds: {train_seeds}")

    set_all_seeds(split_seed, deterministic=True)
    edge_index, X, y, s, masks = load_pokec_z_dataset(
        csv_path=csv_path,
        edge_path=edge_path,
        relationship_path=relationship_path,
        embedding_path=embedding_path,
        standardize=True,
        use_embedding=True,
        predict_attr="I_am_working_in_field",
        sens_attr="region",
        drop_cols=("user_id",),
        split_seed=split_seed,
    )

    if run_baseline_once:
        _ = baseline_logreg(X, y, masks, s)

    results = []

    for train_seed in train_seeds:
        print_header(f"Training with train_seed = {train_seed}")
        set_all_seeds(train_seed, deterministic=True)

        result = train_depthfairgnn_one_seed(
            X, edge_index, y, s, masks,
            hidden=FIXED_PARAMS["hidden"],
            num_layers=FIXED_PARAMS["num_layers"],
            epochs=FIXED_PARAMS["epochs"],
            lr=FIXED_PARAMS["lr"],
            weight_decay=FIXED_PARAMS["weight_decay"],
            dropout=FIXED_PARAMS["dropout"],
            residual_alpha_deep=FIXED_PARAMS["residual_alpha_deep"],
            lambda_weak=FIXED_PARAMS["lambda_weak"],
            lambda_mid=FIXED_PARAMS["lambda_mid"],
            lambda_strong=FIXED_PARAMS["lambda_strong"],
            fairness_warmup=FIXED_PARAMS["fairness_warmup"],
            use_rbf_mmd=FIXED_PARAMS["use_rbf_mmd"],
            mmd_sigma=FIXED_PARAMS["mmd_sigma"],
            pos_weight_multiplier=FIXED_PARAMS["pos_weight_multiplier"],
        )

        row = {
            "train_seed": train_seed,
            "best_epoch": result["best_epoch"],
            "best_val_auc": result["best_val_auc"],
            "ACC": result["test_metrics"]["acc"],
            "SP": result["test_metrics"]["spd"],
            "AUC": result["test_metrics"]["auc"],
            "EO": result["test_metrics"]["eod"],
            "F1": result["test_metrics"]["f1"],
            "Precision": result["test_metrics"]["precision"],
            "Recall": result["test_metrics"]["recall"],
            "PredPosRate": result["test_metrics"]["pred_pos_rate"],
        }
        results.append(row)
        print("Seed result:", row)

    results_df = pd.DataFrame(results)

    summary_df = pd.DataFrame([{
        "ACC_mean": safe_mean(results_df["ACC"]),
        "ACC_std": safe_std(results_df["ACC"]),
        "SP_mean": safe_mean(results_df["SP"]),
        "SP_std": safe_std(results_df["SP"]),
        "AUC_mean": safe_mean(results_df["AUC"]),
        "AUC_std": safe_std(results_df["AUC"]),
        "EO_mean": safe_mean(results_df["EO"]),
        "EO_std": safe_std(results_df["EO"]),
        "F1_mean": safe_mean(results_df["F1"]),
        "F1_std": safe_std(results_df["F1"]),
        "Precision_mean": safe_mean(results_df["Precision"]),
        "Precision_std": safe_std(results_df["Precision"]),
        "Recall_mean": safe_mean(results_df["Recall"]),
        "Recall_std": safe_std(results_df["Recall"]),
        "PredPosRate_mean": safe_mean(results_df["PredPosRate"]),
        "PredPosRate_std": safe_std(results_df["PredPosRate"]),
    }])

    print_header("Per-seed results")
    print(results_df[["train_seed", "ACC", "SP", "AUC", "EO", "F1", "Precision", "Recall", "PredPosRate"]])

    print_header("Final mean/std")
    print(summary_df)

    results_csv = os.path.join(save_dir, "seed_results.csv")
    results_json = os.path.join(save_dir, "seed_results.json")
    summary_csv = os.path.join(save_dir, "summary_mean_std.csv")
    summary_json = os.path.join(save_dir, "summary_mean_std.json")

    results_df.to_csv(results_csv, index=False)
    results_df.to_json(results_json, orient="records", indent=2)
    summary_df.to_csv(summary_csv, index=False)
    summary_df.to_json(summary_json, orient="records", indent=2)

    print(f"\nSaved per-seed results to: {results_csv}")
    print(f"Saved per-seed results json to: {results_json}")
    print(f"Saved summary csv to: {summary_csv}")
    print(f"Saved summary json to: {summary_json}")

    return results_df, summary_df


# =========================================================
# 11) Main
# =========================================================
if __name__ == "__main__":
    FIXED_SPLIT_SEED = 42
    TRAIN_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

    results_df, summary_df = run_fixed_best_params_10_seeds_fixed_split_pokec_z(
        csv_path="/root/autodl-tmp/pokec_z/region_job.csv",
        edge_path="/root/autodl-tmp/pokec_z/region_job_edges.txt",
        relationship_path="/root/autodl-tmp/pokec_z/region_job_relationship.txt",
        embedding_path="/root/autodl-tmp/pokec_z/region_job.embedding",
        save_dir="/root/autodl-tmp/pokec_z/best_parameter_run_fixed_split",
        split_seed=FIXED_SPLIT_SEED,
        train_seeds=TRAIN_SEEDS,
        epochs=300,
        run_baseline_once=True,
    )


Fixed hyperparameters
{'hidden': 64, 'num_layers': 6, 'epochs': 300, 'lr': 0.01, 'weight_decay': 0.0005, 'dropout': 0.4, 'residual_alpha_deep': 0.2, 'lambda_weak': 0.01, 'lambda_mid': 0.05, 'lambda_strong': 0.1, 'fairness_warmup': 100, 'use_rbf_mmd': False, 'mmd_sigma': 1.0, 'pos_weight_multiplier': 1.0}

Evaluation setting
Fixed split seed: 42
Training seeds: [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

Loading POKEC_Z dataset | fixed split seed=42
Embedding matched rows: 67796 / 67796
Num nodes: 67796  Num features: 340  Num edges: 574542
Degree: min/med/max: 1 4.0 331
Isolated nodes: 0
Label distribution: {0: 62298, 1: 5498}
Sensitive distribution: {0: 43962, 1: 23834}
Mask overlaps (should be 0s): {'train∩val': 0, 'train∩test': 0, 'val∩test': 0}
Train count: 33898
Val count: 16949
Test count: 16949
Done in 4.636s
Edges (directed columns): 574542
Edges (undirected unique pairs): 287271

Baseline: Logistic Regression (features-only)
Train: acc=0.712 auc=0.843 f1=0.320 prec=0.198 rec=0.

# Pokec_n

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import time
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn.functional as F
from torch import nn

from torch_geometric.utils import to_undirected, remove_self_loops, coalesce

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, accuracy_score,
    f1_score, precision_score, recall_score
)


# =========================================================
# 0) Reproducibility
# =========================================================
def set_all_seeds(seed: int, deterministic: bool = True):
    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            pass
    else:
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True


# =========================================================
# 1) Small utilities
# =========================================================
def print_header(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def undirected_unique_edge_count(edge_index, num_nodes):
    u = torch.minimum(edge_index[0], edge_index[1])
    v = torch.maximum(edge_index[0], edge_index[1])
    uv = torch.stack([u, v], dim=0)
    uv = coalesce(uv, None, num_nodes, num_nodes)[0]
    return uv.size(1)


def safe_mean(arr):
    arr = np.array(arr, dtype=float)
    return float(np.nanmean(arr))


def safe_std(arr):
    arr = np.array(arr, dtype=float)
    return float(np.nanstd(arr, ddof=1)) if len(arr) > 1 else 0.0


# =========================================================
# 2) Robust edge reader + embedding reader
# =========================================================
def read_and_validate_edges(edge_file, n, make_undirected=True, drop_self_loops=True):
    edges = []

    with open(edge_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 2:
                continue
            try:
                u = int(float(parts[0]))
                v = int(float(parts[1]))
            except ValueError:
                continue
            if 0 <= u < n and 0 <= v < n:
                edges.append([u, v])

    if len(edges) == 0:
        raise ValueError(f"No valid edges found in file: {edge_file}")

    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    if drop_self_loops:
        edge_index, _ = remove_self_loops(edge_index)

    if make_undirected:
        edge_index = to_undirected(edge_index, num_nodes=n)

    edge_index = coalesce(edge_index, None, n, n)[0]
    return edge_index


def read_embedding_file(embedding_path, user_id_series, expected_dim=None):
    user_id_to_idx = {uid: idx for idx, uid in enumerate(user_id_series.tolist())}
    n = len(user_id_series)

    with open(embedding_path, "r", encoding="utf-8", errors="ignore") as f:
        first_line = f.readline().strip().split()
        if len(first_line) == 2:
            _, emb_dim = map(int, first_line)
        else:
            raise ValueError("Embedding header format unexpected. Expected '<num_nodes> <dim>'")

        if expected_dim is not None and emb_dim != expected_dim:
            raise ValueError(f"Embedding dim mismatch: got {emb_dim}, expected {expected_dim}")

        emb = np.zeros((n, emb_dim), dtype=np.float32)
        found = np.zeros(n, dtype=bool)

        for line in f:
            parts = line.strip().split()
            if len(parts) < emb_dim + 1:
                continue
            try:
                raw_uid = int(float(parts[0]))
                vec = np.array(parts[1:1 + emb_dim], dtype=np.float32)
            except ValueError:
                continue

            if raw_uid in user_id_to_idx:
                idx = user_id_to_idx[raw_uid]
                emb[idx] = vec
                found[idx] = True

    print(f"Embedding matched rows: {found.sum()} / {n}")
    return emb


# =========================================================
# 3) Stratified split
# =========================================================
def stratified_masks(y, train_ratio=0.5, val_ratio=0.25, seed=42):
    y_np = y.cpu().numpy()
    N = len(y_np)
    tv_ratio = train_ratio + val_ratio

    sss1 = StratifiedShuffleSplit(
        n_splits=1,
        test_size=1 - tv_ratio,
        random_state=seed
    )
    (tv_idx, te_idx) = next(sss1.split(np.zeros(N), y_np))

    y_tv = y_np[tv_idx]
    sss2 = StratifiedShuffleSplit(
        n_splits=1,
        test_size=val_ratio / tv_ratio,
        random_state=seed
    )
    (tr_rel, va_rel) = next(sss2.split(np.zeros(len(tv_idx)), y_tv))

    tr_idx = tv_idx[tr_rel]
    va_idx = tv_idx[va_rel]

    train_mask = torch.zeros(N, dtype=torch.bool)
    val_mask = torch.zeros(N, dtype=torch.bool)
    test_mask = torch.zeros(N, dtype=torch.bool)

    train_mask[tr_idx] = True
    val_mask[va_idx] = True
    test_mask[te_idx] = True

    return {"train": train_mask, "val": val_mask, "test": test_mask}


# =========================================================
# 4) Loader for Pokec_n
# =========================================================
def load_pokec_n_dataset(
    csv_path="/root/autodl-tmp/pokec_n/region_job_2.csv",
    edge_path="/root/autodl-tmp/pokec_n/region_job_2_edges.txt",
    relationship_path="/root/autodl-tmp/pokec_n/region_job_2_relationship.txt",
    embedding_path="/root/autodl-tmp/pokec_n/region_job_2.embedding",
    standardize=True,
    use_embedding=True,
    predict_attr="I_am_working_in_field",
    sens_attr="region",
    drop_cols=("user_id",),
    split_seed=42,
):
    for p in [csv_path, edge_path, relationship_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Missing file: {p}")
    if use_embedding and not os.path.exists(embedding_path):
        raise FileNotFoundError(f"Missing embedding file: {embedding_path}")

    print_header(f"Loading POKEC_N dataset | fixed split seed={split_seed}")
    t0 = time.time()
    df = pd.read_csv(csv_path)

    if predict_attr not in df.columns:
        raise ValueError(f"predict_attr '{predict_attr}' not found in CSV")
    if sens_attr not in df.columns:
        raise ValueError(f"sens_attr '{sens_attr}' not found in CSV")
    if "user_id" not in df.columns:
        raise ValueError("Pokec_n CSV must contain 'user_id' column")

    raw_label = pd.to_numeric(df[predict_attr], errors="coerce").fillna(-1).astype(int).values
    labels = (raw_label > 0).astype(int)

    sens = pd.to_numeric(df[sens_attr], errors="coerce").fillna(0).astype(int).values
    if not set(np.unique(sens)).issubset({0, 1}):
        raise ValueError(f"Sensitive attribute must be binary {{0,1}}, got {np.unique(sens)}")

    labels_t = torch.tensor(labels, dtype=torch.long)
    sens_t = torch.tensor(sens, dtype=torch.long)

    masks = stratified_masks(labels_t, train_ratio=0.5, val_ratio=0.25, seed=split_seed)

    keep_cols = [c for c in df.columns if c not in set(drop_cols) | {predict_attr, sens_attr}]
    X_df = df[keep_cols].copy()

    cat_cols = [c for c in X_df.columns if X_df[c].dtype == "object"]
    if cat_cols:
        X_df = pd.get_dummies(X_df, columns=cat_cols, drop_first=True)

    for c in X_df.columns:
        X_df[c] = pd.to_numeric(X_df[c], errors="coerce")
    X_df = X_df.fillna(0)

    X_np = X_df.to_numpy(dtype=np.float32)

    if standardize:
        scaler = StandardScaler()
        train_idx = masks["train"].cpu().numpy()
        scaler.fit(X_np[train_idx])
        X_np = scaler.transform(X_np).astype(np.float32)

    if use_embedding:
        emb_np = read_embedding_file(
            embedding_path=embedding_path,
            user_id_series=df["user_id"],
            expected_dim=None
        )
        X_np = np.concatenate([X_np, emb_np], axis=1).astype(np.float32)

    features = torch.tensor(X_np, dtype=torch.float32)

    n = features.size(0)
    edge_index_1 = read_and_validate_edges(edge_path, n, make_undirected=True, drop_self_loops=True)
    edge_index_2 = read_and_validate_edges(relationship_path, n, make_undirected=True, drop_self_loops=True)

    edge_index = torch.cat([edge_index_1, edge_index_2], dim=1)
    edge_index = coalesce(edge_index, None, n, n)[0]

    print("Num nodes:", n, " Num features:", features.size(1), " Num edges:", edge_index.size(1))
    deg = torch.bincount(edge_index[0], minlength=n).cpu().numpy()
    print("Degree: min/med/max:", int(deg.min()), float(np.median(deg)), int(deg.max()))
    print("Isolated nodes:", int((deg == 0).sum()))

    u_lbl, c_lbl = np.unique(labels, return_counts=True)
    print("Label distribution:", dict(zip(u_lbl.tolist(), c_lbl.tolist())))

    u_s, c_s = np.unique(sens, return_counts=True)
    print("Sensitive distribution:", dict(zip(u_s.tolist(), c_s.tolist())))

    overlap = {
        "train∩val": int((masks["train"] & masks["val"]).sum()),
        "train∩test": int((masks["train"] & masks["test"]).sum()),
        "val∩test": int((masks["val"] & masks["test"]).sum()),
    }
    print("Mask overlaps (should be 0s):", overlap)

    print("Train count:", int(masks["train"].sum()))
    print("Val count:", int(masks["val"].sum()))
    print("Test count:", int(masks["test"].sum()))

    print(f"Done in {time.time()-t0:.3f}s")
    print(f"Edges (directed columns): {edge_index.size(1)}")
    print(f"Edges (undirected unique pairs): {undirected_unique_edge_count(edge_index, n)}")

    return edge_index, features, labels_t, sens_t, masks


# =========================================================
# 5) Graph normalization
# =========================================================
def build_gcn_norm(edge_index, num_nodes, add_self_loops=True, device=None):
    if device is None:
        device = edge_index.device

    row, col = edge_index
    if add_self_loops:
        self_loops = torch.arange(num_nodes, device=device)
        self_loops = torch.stack([self_loops, self_loops], dim=0)
        edge_index = torch.cat([edge_index, self_loops], dim=1)
        edge_index = coalesce(edge_index, None, num_nodes, num_nodes)[0]
        row, col = edge_index

    deg = torch.bincount(row, minlength=num_nodes).float()
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[torch.isinf(deg_inv_sqrt)] = 0.0

    norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]
    return edge_index, norm


def spmm_mean(edge_index, edge_weight, x):
    row, col = edge_index
    out = torch.zeros_like(x)
    msg = x[col] * edge_weight.unsqueeze(-1)
    out.index_add_(0, row, msg)
    return out


# =========================================================
# 6) DepthFairGNN layers
# =========================================================
class DepthFairLayer(nn.Module):
    def __init__(self, in_dim, out_dim, dropout=0.2, use_bn=True):
        super().__init__()
        self.lin = nn.Linear(in_dim, out_dim, bias=False)
        self.use_bn = use_bn
        self.bn = nn.BatchNorm1d(out_dim) if use_bn else nn.Identity()
        self.do = nn.Dropout(dropout)
        self.res_proj = nn.Linear(in_dim, out_dim, bias=False) if in_dim != out_dim else None

    def forward(self, x, edge_index, edge_weight, identity=None, residual_alpha=0.0):
        h_in = x
        x = self.lin(x)
        h = spmm_mean(edge_index, edge_weight, x)

        h = h + (self.res_proj(h_in) if self.res_proj is not None else h_in)

        if identity is not None and residual_alpha > 0:
            h = h + residual_alpha * identity

        h = self.bn(h)
        h = F.relu(h)
        h = self.do(h)
        return h


class DepthFairGNN(nn.Module):
    def __init__(
        self,
        in_dim,
        hidden=128,
        out_dim=2,
        num_layers=6,
        dropout=0.2,
        residual_alpha_deep=0.2,
    ):
        super().__init__()
        assert num_layers >= 2, "Need at least 2 layers."

        self.num_layers = num_layers
        self.residual_alpha_deep = residual_alpha_deep

        self.input_proj = nn.Linear(in_dim, hidden, bias=False)

        self.layers = nn.ModuleList()
        self.layers.append(DepthFairLayer(in_dim, hidden, dropout=dropout))
        for _ in range(num_layers - 2):
            self.layers.append(DepthFairLayer(hidden, hidden, dropout=dropout))

        self.cls = nn.Sequential(
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, out_dim)
        )

    def forward(self, x, edge_index, edge_weight):
        hidden_states = []
        identity = self.input_proj(x)

        h = x
        for i, layer in enumerate(self.layers):
            layer_num = i + 1
            alpha = self.residual_alpha_deep if layer_num >= 5 else 0.0
            h = layer(h, edge_index, edge_weight, identity=identity, residual_alpha=alpha)
            hidden_states.append(h)

        logits = self.cls(h)
        return logits, hidden_states


# =========================================================
# 7) Fairness loss + metrics
# =========================================================
def linear_mmd(x0, x1):
    if x0.size(0) == 0 or x1.size(0) == 0:
        device = x0.device if x0.numel() > 0 else x1.device
        return torch.tensor(0.0, device=device)
    return ((x0.mean(dim=0) - x1.mean(dim=0)) ** 2).mean()


def rbf_mmd(x0, x1, sigma=1.0):
    if x0.size(0) < 2 or x1.size(0) < 2:
        device = x0.device if x0.numel() > 0 else x1.device
        return torch.tensor(0.0, device=device)

    xx = torch.cdist(x0, x0, p=2).pow(2)
    yy = torch.cdist(x1, x1, p=2).pow(2)
    xy = torch.cdist(x0, x1, p=2).pow(2)

    k_xx = torch.exp(-xx / (2 * sigma * sigma))
    k_yy = torch.exp(-yy / (2 * sigma * sigma))
    k_xy = torch.exp(-xy / (2 * sigma * sigma))
    return k_xx.mean() + k_yy.mean() - 2 * k_xy.mean()


def depth_schedule(num_layers, lambda_weak=0.01, lambda_mid=0.05, lambda_strong=0.10):
    weights = []
    for l in range(1, num_layers + 1):
        if l <= 2:
            weights.append(lambda_weak)
        elif l <= 4:
            weights.append(lambda_mid)
        else:
            weights.append(lambda_strong)
    return weights


def depth_fairness_loss(hidden_states, sensitive, train_mask, layer_weights, use_rbf=False, sigma=1.0):
    loss = torch.tensor(0.0, device=hidden_states[0].device)
    for l, h in enumerate(hidden_states, start=1):
        h_train = h[train_mask]
        s_train = sensitive[train_mask]
        h0 = h_train[s_train == 0]
        h1 = h_train[s_train == 1]

        if use_rbf:
            fair_l = rbf_mmd(h0, h1, sigma=sigma)
        else:
            fair_l = linear_mmd(h0, h1)

        loss = loss + layer_weights[l - 1] * fair_l
    return loss


@torch.no_grad()
def metrics(logits, y, sensitive):
    pred = logits.argmax(dim=-1)
    acc = (pred == y).float().mean().item()

    p1 = F.softmax(logits, dim=-1)[:, 1].cpu().numpy()
    y_np = y.cpu().numpy()
    s_np = sensitive.cpu().numpy()
    pred_np = pred.cpu().numpy()

    try:
        auc = roc_auc_score(y_np, p1)
    except ValueError:
        auc = float("nan")

    f1 = f1_score(y_np, pred_np, zero_division=0)
    precision = precision_score(y_np, pred_np, zero_division=0)
    recall = recall_score(y_np, pred_np, zero_division=0)
    pred_pos_rate = float((pred_np == 1).mean())

    s0 = s_np == 0
    s1 = s_np == 1
    p1_mean_s0 = p1[s0].mean() if s0.any() else np.nan
    p1_mean_s1 = p1[s1].mean() if s1.any() else np.nan
    spd = abs(p1_mean_s0 - p1_mean_s1)

    pred_pos = (pred_np == 1)
    y_pos = (y_np == 1)

    def tpr(mask):
        denom = (y_pos & mask).sum()
        return (pred_pos & y_pos & mask).sum() / denom if denom > 0 else np.nan

    tpr0 = tpr(s0)
    tpr1 = tpr(s1)
    eod = abs(tpr0 - tpr1) if np.isfinite(tpr0) and np.isfinite(tpr1) else np.nan

    return {
        "acc": acc,
        "auc": auc,
        "f1": f1,
        "precision": precision,
        "recall": recall,
        "pred_pos_rate": pred_pos_rate,
        "spd": spd,
        "eod": eod,
    }


# =========================================================
# 8) Baseline
# =========================================================
def baseline_logreg(features, labels, masks, sensitive):
    print_header("Baseline: Logistic Regression (features-only)")
    X = features.cpu().numpy()
    y = labels.cpu().numpy()
    s_np = sensitive.cpu().numpy()

    tr = masks["train"].cpu().numpy()
    va = masks["val"].cpu().numpy()
    te = masks["test"].cpu().numpy()

    def fairness_metrics(prob, pred, y_true, sens):
        s0, s1 = sens == 0, sens == 1
        p1_s0 = prob[s0].mean() if s0.any() else np.nan
        p1_s1 = prob[s1].mean() if s1.any() else np.nan
        spd = abs(p1_s0 - p1_s1)

        y_pos = y_true == 1
        def tpr(mask):
            denom = (y_pos & mask).sum()
            return (pred & y_pos & mask).sum() / denom if denom > 0 else np.nan

        tpr0 = tpr(s0)
        tpr1 = tpr(s1)
        eod = abs(tpr0 - tpr1) if np.isfinite(tpr0) and np.isfinite(tpr1) else np.nan
        return spd, eod

    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(X[tr], y[tr])

    for name, msk in [("Train", tr), ("Val", va), ("Test", te)]:
        prob = clf.predict_proba(X[msk])[:, 1]
        pred = (prob >= 0.5).astype(int)
        acc = accuracy_score(y[msk], pred)
        try:
            auc = roc_auc_score(y[msk], prob)
        except ValueError:
            auc = float("nan")
        f1 = f1_score(y[msk], pred, zero_division=0)
        precision = precision_score(y[msk], pred, zero_division=0)
        recall = recall_score(y[msk], pred, zero_division=0)
        pred_pos_rate = float((pred == 1).mean())
        spd, eod = fairness_metrics(prob, pred, y[msk], s_np[msk])
        print(
            f"{name}: acc={acc:.3f} auc={auc:.3f} f1={f1:.3f} "
            f"prec={precision:.3f} rec={recall:.3f} pred1={pred_pos_rate:.3f} "
            f"spd={spd:.3f} eod={eod:.3f}"
        )

    return clf


# =========================================================
# 9) Train one seed
# =========================================================
def train_depthfairgnn_one_seed(
    x, edge_index, y, sensitive, masks,
    hidden=128,
    num_layers=6,
    epochs=300,
    lr=0.01,
    weight_decay=1e-4,
    dropout=0.2,
    residual_alpha_deep=0.2,
    lambda_weak=0.01,
    lambda_mid=0.05,
    lambda_strong=0.10,
    fairness_warmup=50,
    use_rbf_mmd=False,
    mmd_sigma=1.0,
    pos_weight_multiplier=1.0,
):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    x = x.to(device)
    edge_index = edge_index.to(device)
    y = y.to(device)
    sensitive = sensitive.to(device)

    train_mask = masks["train"].to(device)
    val_mask = masks["val"].to(device)
    test_mask = masks["test"].to(device)

    edge_index_norm, edge_weight = build_gcn_norm(
        edge_index, num_nodes=x.size(0), add_self_loops=True, device=device
    )

    train_y = y[train_mask]
    num_pos = int((train_y == 1).sum().item())
    num_neg = int((train_y == 0).sum().item())
    base_pos_weight = num_neg / max(num_pos, 1)
    final_pos_weight = base_pos_weight * pos_weight_multiplier

    cw = torch.tensor(
        [1.0, final_pos_weight],
        dtype=torch.float32,
        device=device
    )

    print_header("CLASS WEIGHT INFO")
    print(f"train neg = {num_neg}")
    print(f"train pos = {num_pos}")
    print(f"base_pos_weight = {base_pos_weight:.4f}")
    print(f"pos_weight_multiplier = {pos_weight_multiplier:.4f}")
    print(f"final positive class weight = {final_pos_weight:.4f}")

    model = DepthFairGNN(
        in_dim=x.size(1),
        hidden=hidden,
        out_dim=2,
        num_layers=num_layers,
        dropout=dropout,
        residual_alpha_deep=residual_alpha_deep
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    base_layer_weights = depth_schedule(
        num_layers=num_layers,
        lambda_weak=lambda_weak,
        lambda_mid=lambda_mid,
        lambda_strong=lambda_strong
    )

    best_val_score = -1e18
    best_state = None
    best_epoch = 0
    best_val_metrics = None
    best_test_metrics = None

    for ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()

        logits, hidden_states = model(x, edge_index_norm, edge_weight)

        alpha = min(1.0, ep / float(fairness_warmup)) if fairness_warmup > 0 else 1.0
        layer_weights = [alpha * w for w in base_layer_weights]

        ce = F.cross_entropy(logits[train_mask], y[train_mask], weight=cw)
        fair = depth_fairness_loss(
            hidden_states=hidden_states,
            sensitive=sensitive,
            train_mask=train_mask,
            layer_weights=layer_weights,
            use_rbf=use_rbf_mmd,
            sigma=mmd_sigma
        )
        loss = ce + fair
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            logits_eval, _ = model(x, edge_index_norm, edge_weight)
            trm = metrics(logits_eval[train_mask], y[train_mask], sensitive[train_mask])
            valm = metrics(logits_eval[val_mask], y[val_mask], sensitive[val_mask])
            tem = metrics(logits_eval[test_mask], y[test_mask], sensitive[test_mask])

            val_score = valm["auc"] + 0.2 * valm["acc"] - 0.1 * valm["pred_pos_rate"]

            if val_score > best_val_score:
                best_val_score = val_score
                best_epoch = ep
                best_val_metrics = valm.copy()
                best_test_metrics = tem.copy()
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        if ep % 10 == 0 or ep == 1:
            print(
                f"[{ep:03d}] loss={loss.item():.4f} ce={ce.item():.4f} fair={fair.item():.4f} "
                f"train(acc={trm['acc']:.3f}, auc={trm['auc']:.3f}, f1={trm['f1']:.3f}, rec={trm['recall']:.3f}, pred1={trm['pred_pos_rate']:.3f}) "
                f"val(acc={valm['acc']:.3f}, auc={valm['auc']:.3f}, f1={valm['f1']:.3f}, rec={valm['recall']:.3f}, pred1={valm['pred_pos_rate']:.3f}) "
                f"test(acc={tem['acc']:.3f}, auc={tem['auc']:.3f}, f1={tem['f1']:.3f}, rec={tem['recall']:.3f}, pred1={tem['pred_pos_rate']:.3f})"
            )

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    with torch.no_grad():
        logits, hidden_states = model(x, edge_index_norm, edge_weight)
        train_metrics = metrics(logits[train_mask], y[train_mask], sensitive[train_mask])
        val_metrics = metrics(logits[val_mask], y[val_mask], sensitive[val_mask])
        test_metrics = metrics(logits[test_mask], y[test_mask], sensitive[test_mask])

    print_header("BEST MODEL SUMMARY")
    print(f"Best epoch: {best_epoch}")
    print(f"Best val score: {best_val_score:.6f}")

    print_header("FINAL TRAIN METRICS")
    print(train_metrics)

    print_header("FINAL VAL METRICS")
    print(val_metrics)

    print_header("FINAL TEST METRICS")
    print(test_metrics)

    return {
        "best_epoch": best_epoch,
        "best_val_auc": best_val_score,
        "train_metrics": train_metrics,
        "val_metrics": val_metrics,
        "test_metrics": test_metrics,
    }


# =========================================================
# 10) Multi-seed runner with fixed split
# =========================================================
def run_fixed_best_params_10_seeds_fixed_split_pokec_n(
    csv_path,
    edge_path,
    relationship_path,
    embedding_path,
    save_dir,
    split_seed=42,
    train_seeds=None,
    epochs=300,
    run_baseline_once=True,
):
    if train_seeds is None:
        train_seeds = list(range(42, 52))

    os.makedirs(save_dir, exist_ok=True)

    FIXED_PARAMS = {
        "hidden": 64,
        "num_layers": 6,
        "epochs": epochs,
        "lr": 0.01,
        "weight_decay": 5e-4,
        "dropout": 0.4,
        "residual_alpha_deep": 0.2,
        "lambda_weak": 0.01,
        "lambda_mid": 0.05,
        "lambda_strong": 0.10,
        "fairness_warmup": 100,
        "use_rbf_mmd": False,
        "mmd_sigma": 1.0,
        "pos_weight_multiplier": 1.0,
    }

    print_header("Fixed hyperparameters")
    print(FIXED_PARAMS)

    print_header("Evaluation setting")
    print(f"Fixed split seed: {split_seed}")
    print(f"Training seeds: {train_seeds}")

    set_all_seeds(split_seed, deterministic=True)
    edge_index, X, y, s, masks = load_pokec_n_dataset(
        csv_path=csv_path,
        edge_path=edge_path,
        relationship_path=relationship_path,
        embedding_path=embedding_path,
        standardize=True,
        use_embedding=True,
        predict_attr="I_am_working_in_field",
        sens_attr="region",
        drop_cols=("user_id",),
        split_seed=split_seed,
    )

    if run_baseline_once:
        _ = baseline_logreg(X, y, masks, s)

    results = []

    for train_seed in train_seeds:
        print_header(f"Training with train_seed = {train_seed}")
        set_all_seeds(train_seed, deterministic=True)

        result = train_depthfairgnn_one_seed(
            X, edge_index, y, s, masks,
            hidden=FIXED_PARAMS["hidden"],
            num_layers=FIXED_PARAMS["num_layers"],
            epochs=FIXED_PARAMS["epochs"],
            lr=FIXED_PARAMS["lr"],
            weight_decay=FIXED_PARAMS["weight_decay"],
            dropout=FIXED_PARAMS["dropout"],
            residual_alpha_deep=FIXED_PARAMS["residual_alpha_deep"],
            lambda_weak=FIXED_PARAMS["lambda_weak"],
            lambda_mid=FIXED_PARAMS["lambda_mid"],
            lambda_strong=FIXED_PARAMS["lambda_strong"],
            fairness_warmup=FIXED_PARAMS["fairness_warmup"],
            use_rbf_mmd=FIXED_PARAMS["use_rbf_mmd"],
            mmd_sigma=FIXED_PARAMS["mmd_sigma"],
            pos_weight_multiplier=FIXED_PARAMS["pos_weight_multiplier"],
        )

        row = {
            "train_seed": train_seed,
            "best_epoch": result["best_epoch"],
            "best_val_auc": result["best_val_auc"],
            "ACC": result["test_metrics"]["acc"],
            "SP": result["test_metrics"]["spd"],
            "AUC": result["test_metrics"]["auc"],
            "EO": result["test_metrics"]["eod"],
            "F1": result["test_metrics"]["f1"],
            "Precision": result["test_metrics"]["precision"],
            "Recall": result["test_metrics"]["recall"],
            "PredPosRate": result["test_metrics"]["pred_pos_rate"],
        }
        results.append(row)
        print("Seed result:", row)

    results_df = pd.DataFrame(results)

    summary_df = pd.DataFrame([{
        "ACC_mean": safe_mean(results_df["ACC"]),
        "ACC_std": safe_std(results_df["ACC"]),
        "SP_mean": safe_mean(results_df["SP"]),
        "SP_std": safe_std(results_df["SP"]),
        "AUC_mean": safe_mean(results_df["AUC"]),
        "AUC_std": safe_std(results_df["AUC"]),
        "EO_mean": safe_mean(results_df["EO"]),
        "EO_std": safe_std(results_df["EO"]),
        "F1_mean": safe_mean(results_df["F1"]),
        "F1_std": safe_std(results_df["F1"]),
        "Precision_mean": safe_mean(results_df["Precision"]),
        "Precision_std": safe_std(results_df["Precision"]),
        "Recall_mean": safe_mean(results_df["Recall"]),
        "Recall_std": safe_std(results_df["Recall"]),
        "PredPosRate_mean": safe_mean(results_df["PredPosRate"]),
        "PredPosRate_std": safe_std(results_df["PredPosRate"]),
    }])

    print_header("Per-seed results")
    print(results_df[["train_seed", "ACC", "SP", "AUC", "EO", "F1", "Precision", "Recall", "PredPosRate"]])

    print_header("Final mean/std")
    print(summary_df)

    results_csv = os.path.join(save_dir, "seed_results.csv")
    results_json = os.path.join(save_dir, "seed_results.json")
    summary_csv = os.path.join(save_dir, "summary_mean_std.csv")
    summary_json = os.path.join(save_dir, "summary_mean_std.json")

    results_df.to_csv(results_csv, index=False)
    results_df.to_json(results_json, orient="records", indent=2)
    summary_df.to_csv(summary_csv, index=False)
    summary_df.to_json(summary_json, orient="records", indent=2)

    print(f"\nSaved per-seed results to: {results_csv}")
    print(f"Saved per-seed results json to: {results_json}")
    print(f"Saved summary csv to: {summary_csv}")
    print(f"Saved summary json to: {summary_json}")

    return results_df, summary_df


# =========================================================
# 11) Main
# =========================================================
if __name__ == "__main__":
    FIXED_SPLIT_SEED = 42
    TRAIN_SEEDS = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

    results_df, summary_df = run_fixed_best_params_10_seeds_fixed_split_pokec_n(
        csv_path="/root/autodl-tmp/pokec_n/region_job_2.csv",
        edge_path="/root/autodl-tmp/pokec_n/region_job_2_edges.txt",
        relationship_path="/root/autodl-tmp/pokec_n/region_job_2_relationship.txt",
        embedding_path="/root/autodl-tmp/pokec_n/region_job_2.embedding",
        save_dir="/root/autodl-tmp/pokec_n/best_parameter_run_fixed_split",
        split_seed=FIXED_SPLIT_SEED,
        train_seeds=TRAIN_SEEDS,
        epochs=300,
        run_baseline_once=True,
    )


Fixed hyperparameters
{'hidden': 64, 'num_layers': 6, 'epochs': 300, 'lr': 0.01, 'weight_decay': 0.0005, 'dropout': 0.4, 'residual_alpha_deep': 0.2, 'lambda_weak': 0.01, 'lambda_mid': 0.05, 'lambda_strong': 0.1, 'fairness_warmup': 100, 'use_rbf_mmd': False, 'mmd_sigma': 1.0, 'pos_weight_multiplier': 1.0}

Evaluation setting
Fixed split seed: 42
Training seeds: [42, 43, 44, 45, 46, 47, 48, 49, 50, 51]

Loading POKEC_N dataset | fixed split seed=42
Embedding matched rows: 66569 / 66569
Num nodes: 66569  Num features: 329  Num edges: 226240
Degree: min/med/max: 1 3.0 201
Isolated nodes: 0
Label distribution: {0: 62282, 1: 4287}
Sensitive distribution: {0: 47338, 1: 19231}
Mask overlaps (should be 0s): {'train∩val': 0, 'train∩test': 0, 'val∩test': 0}
Train count: 33284
Val count: 16642
Test count: 16643
Done in 4.425s
Edges (directed columns): 226240
Edges (undirected unique pairs): 113120

Baseline: Logistic Regression (features-only)
Train: acc=0.710 auc=0.845 f1=0.274 prec=0.164 rec=0.